# Factors of regional population decline in Japan

JAIST K417EJ capstone. This notebook runs the whole analysis on Google Colab: it downloads data from the
e-Stat API (Social and Demographic Statistics System, SSDS), builds 5-year panels for 47 prefectures and
1,741 municipalities, and answers the four research questions:

1. When did regional decline start, and does it line up with historical events? (step 6)
2. Which factors contribute most? (steps 7–8)
3. Do the main factors change over time? (step 9)
4. How far will decline go by 2025 and 2030? (step 8b)

**Before you run:** add your e-Stat application ID as a Colab secret. Click the key icon in the left
sidebar, add a secret named `ESTAT_APP_ID`, and switch on *Notebook access*. If no secret is found, the
setup cell asks for the key instead.

Then use **Runtime → Run all**. The whole run takes about 10–15 minutes, most of it in the data download
(step 4) and the bootstrap feature selection (step 7).

## Setup

In [ ]:
%pip install -q pyarrow python-dotenv statsmodels scikit-learn

import os, pathlib
ROOT = pathlib.Path(os.environ.get('PROJECT_DIR', '/content/depopulation-Japan'))
for d in ['src', 'config', 'data/raw/meta', 'data/processed', 'reports/figures']:
    (ROOT / d).mkdir(parents=True, exist_ok=True)
os.chdir(ROOT)

key = os.environ.get('ESTAT_APP_ID')
if not key:
  try:
    from google.colab import userdata
    key = userdata.get('ESTAT_APP_ID')
  except Exception:
    pass
if not key:
    from getpass import getpass
    key = getpass('e-Stat application ID: ')
(ROOT / '.env').write_text(f'ESTAT_APP_ID={key.strip()}\n')
os.environ['ESTAT_APP_ID'] = key.strip()
import subprocess, sys

def run(script, quiet=False):
    """Run one pipeline step with this notebook's Python; stop with an error if it fails."""
    r = subprocess.run([sys.executable, f'src/{script}'], capture_output=True, text=True)
    if not quiet or r.returncode:
        print(r.stdout[-6000:])
    if r.returncode:
        print(r.stderr[-6000:])
        raise RuntimeError(f'{script} failed (see the error above)')
    print(f'{script}: done')

print('Project folder:', ROOT)

## Project code

The next cells write the pipeline's source files into the Colab session (`/content/depopulation-Japan/src`).
They're the same files as in the project repository. You don't need to read them to run the notebook.

In [ ]:
%%writefile config/variables.csv
name,code,level,theme,core,description
population,A1101,pref,target,1,Total population
births,A4101,pref,target,1,Births
deaths,A4200,pref,target,1,Deaths
moves_in,A5101,pref,target,1,Moves in (Japanese nationals)
moves_out,A5102,pref,target,1,Moves out (Japanese nationals)
pop_65plus,A1303,pref,demography,1,Population aged 65+
female_20_24,A120502,pref,demography,1,Women aged 20-24
female_25_29,A120602,pref,demography,1,Women aged 25-29
female_30_34,A120702,pref,demography,1,Women aged 30-34
female_35_39,A120802,pref,demography,1,Women aged 35-39
tfr,#A05203,pref,demography,0,Total fertility rate
unmarried_f_30_34,A1601042,pref,demography,0,Unmarried women aged 30-34
taxable_income,C120110,pref,economy,1,Taxable income (thousand yen)
taxpayers,C120120,pref,economy,1,Number of taxpayers (income levy)
unemployment_rate,#F01301,pref,economy,0,Unemployment rate (%)
did_population,A1801,pref,urban,1,Population in densely inhabited districts
vacancy_rate,#H01405,pref,housing,1,Vacant dwelling ratio (%)
land_price_res,C5401,pref,housing,0,Residential land standard price (yen/m2)
univ_entrants_other_pref,E470202,pref,education,1,University entrants from this prefecture's high schools going to another prefecture
univ_entrants_total,E470205,pref,education,1,University entrants from this prefecture's high schools
univ_capacity_index,#E0610202,pref,education,0,University capacity index
doctors,I6100,pref,health,1,Number of doctors
fiscal_strength,D2101,pref,finance,1,Fiscal strength index (prefectural finance)
snow_days,B4107,pref,environment,0,Days of snowfall per year
population,A1101,muni,target,1,Total population
births,A4101,muni,target,1,Births
deaths,A4200,muni,target,1,Deaths
moves_in,A5103,muni,target,0,Moves in (2018+ only)
moves_out,A5104,muni,target,0,Moves out (2018+ only)
pop_65plus,A1303,muni,demography,1,Population aged 65+
female_15_64,A130202,muni,demography,0,Women aged 15-64 (no 20-39 band at municipal level)
taxable_income,C120110,muni,economy,1,Taxable income (thousand yen)
taxpayers,C120120,muni,economy,1,Number of taxpayers (income levy)
unemployment_rate,#F01301,muni,economy,0,Unemployment rate (%)
did_population,A1801,muni,urban,1,Population in densely inhabited districts (missing = no DID)
vacancy_rate,#H01405,muni,housing,1,Vacant dwelling ratio (%)
land_price_res,C5401,muni,housing,0,Residential land standard price (yen/m2)
doctors,I6100,muni,health,1,Number of doctors
fiscal_strength,D2201,muni,finance,1,Fiscal strength index (municipal finance)
job_openings_ratio,#F03103,pref,economy,1,Active job openings-to-applicants ratio (long series; F310301 only covers 2015+)
hs_grads_jobs_other_pref,#F03302,pref,education,1,Share of high-school graduates taking a job in another prefecture (%)
local_univ_entry_share,#E0940302,pref,education,0,Share of university entrants from local high schools entering a local university (%)

In [ ]:
%%writefile src/estat.py
"""Minimal e-Stat API v3 client with on-disk caching of metadata."""

from __future__ import annotations

import json
import os
import time
from pathlib import Path

import pandas as pd
import requests
from dotenv import load_dotenv

ROOT = Path(__file__).resolve().parent.parent
META_DIR = ROOT / "data" / "raw" / "meta"
BASE = "https://api.e-stat.go.jp/rest/3.0/app/json"

load_dotenv(ROOT / ".env")


def _as_list(x) -> list:
    """e-Stat returns a dict instead of a one-element list; normalize."""
    if x is None:
        return []
    return [x] if isinstance(x, dict) else list(x)


def _get(endpoint: str, **params) -> dict:
    params["appId"] = os.environ["ESTAT_APP_ID"]
    for attempt in range(4):
        try:
            r = requests.get(f"{BASE}/{endpoint}", params=params, timeout=300)
            r.raise_for_status()
            return r.json()
        except (requests.RequestException, ValueError):
            if attempt == 3:
                raise
            time.sleep(2 ** attempt * 5)
    raise RuntimeError("unreachable")


def get_meta(stats_data_id: str, refresh: bool = False) -> dict:
    """Return METADATA_INF for a table, cached in data/raw/meta/."""
    META_DIR.mkdir(parents=True, exist_ok=True)
    path = META_DIR / f"{stats_data_id}.json"
    if refresh or not path.exists():
        path.write_text(json.dumps(_get("getMetaInfo", statsDataId=stats_data_id), ensure_ascii=False))
    j = json.loads(path.read_text())
    return j["GET_META_INFO"]["METADATA_INF"]


def classes(stats_data_id: str, class_id: str) -> pd.DataFrame:
    """One classification (e.g. 'cat01', 'area', 'time') as a DataFrame of its @-attributes."""
    for c in _as_list(get_meta(stats_data_id)["CLASS_INF"]["CLASS_OBJ"]):
        if c["@id"] == class_id:
            return pd.DataFrame(_as_list(c["CLASS"])).rename(columns=lambda s: s.lstrip("@"))
    raise KeyError(f"{class_id} not in {stats_data_id}")


def get_data(stats_data_id: str, cat01: list[str]) -> pd.DataFrame:
    """Download values for the given item codes, following pagination.

    Returns long format: code, area_code, time_code, value (float, NaN if not a number),
    raw (original string, keeps symbols like '***'), annotation.
    """
    rows: list[dict] = []
    start = 1
    while True:
        j = _get(
            "getStatsData",
            statsDataId=stats_data_id,
            cdCat01=",".join(cat01),
            startPosition=start,
            limit=100000,
            metaGetFlg="N",
        )["GET_STATS_DATA"]
        if j["RESULT"]["STATUS"] not in (0, 1):  # 1 = no data for this filter
            raise RuntimeError(f"{stats_data_id}: {j['RESULT']}")
        sd = j.get("STATISTICAL_DATA")
        if not sd or "DATA_INF" not in sd:
            break
        rows += _as_list(sd["DATA_INF"]["VALUE"])
        nxt = sd["RESULT_INF"].get("NEXT_KEY")
        if not nxt:
            break
        start = nxt
    if not rows:
        return pd.DataFrame(columns=["code", "area_code", "time_code", "value", "raw", "annotation"])
    df = pd.DataFrame(rows)
    out = pd.DataFrame(
        {
            "code": df["@cat01"],
            "area_code": df["@area"],
            "time_code": df["@time"],
            "raw": df["$"],
            "annotation": df.get("@anno"),
        }
    )
    out["value"] = pd.to_numeric(out["raw"].str.replace(",", ""), errors="coerce")
    return out

In [ ]:
%%writefile src/download.py
"""Step 4: download SSDS variables listed in config/variables.csv.

Usage:
    python src/download.py              # all levels
    python src/download.py --level pref
    python src/download.py --refresh    # re-download even if output exists

Output (long format, one row per variable x area x year):
    data/raw/ssds_pref.parquet, data/raw/ssds_muni.parquet
    data/raw/ssds_manifest.json  (source table per variable, download date, coverage)
"""

from __future__ import annotations

import argparse
import json
from datetime import date

import pandas as pd

import estat

# SSDS table series searched for each level (table suffixes 01..13 = themes A..M).
# Municipal series are the merger-adjusted ones (廃置分合処理済), see docs/datasets.md.
SERIES = {
    "pref": [f"00000101{i:02d}" for i in range(1, 14)] + [f"00000102{i:02d}" for i in range(1, 14)],
    "muni": [f"00000202{i:02d}" for i in range(1, 12)] + [f"00000203{i:02d}" for i in range(1, 12)],
}
RAW = estat.ROOT / "data" / "raw"


def item_index(level: str) -> dict[str, tuple[str, str, str]]:
    """Map item code (name prefix, e.g. 'A1101', '#A05203') -> (statsDataId, cat01 code, Japanese name)."""
    idx: dict[str, tuple[str, str, str]] = {}
    for sid in SERIES[level]:
        for _, row in estat.classes(sid, "cat01").iterrows():
            code = row["name"].split("_", 1)[0]
            idx.setdefault(code, (sid, row["code"], row["name"]))
    return idx


def area_table(sid: str) -> pd.DataFrame:
    a = estat.classes(sid, "area")
    cols = {"code": "area_code", "name": "area_name"}
    if "level" in a:
        cols["level"] = "area_level"
    return a[list(cols)].rename(columns=cols)


def download_level(level: str) -> tuple[pd.DataFrame, list[dict]]:
    spec = pd.read_csv(estat.ROOT / "config" / "variables.csv")
    spec = spec[spec["level"] == level]
    idx = item_index(level)

    missing = [c for c in spec["code"] if c not in idx]
    if missing:
        print(f"[{level}] WARNING: codes not found in SSDS, skipped: {missing}")
    spec = spec[spec["code"].isin(idx)].copy()
    spec["sid"] = spec["code"].map(lambda c: idx[c][0])
    spec["cat01"] = spec["code"].map(lambda c: idx[c][1])
    spec["name_ja"] = spec["code"].map(lambda c: idx[c][2])

    frames = []
    for sid, g in spec.groupby("sid"):
        print(f"[{level}] {sid}: {', '.join(g['name'])}")
        df = estat.get_data(sid, list(g["cat01"]))
        df = df.merge(g[["cat01", "name"]], left_on="code", right_on="cat01").drop(columns=["cat01", "code"])
        df = df.merge(area_table(sid), on="area_code", how="left")
        df["source_table"] = sid
        frames.append(df)

    out = pd.concat(frames, ignore_index=True)
    out["year"] = out["time_code"].str[:4].astype(int)
    out = out.rename(columns={"name": "variable"})
    out = out[["variable", "area_code", "area_name", *(["area_level"] if "area_level" in out else []),
               "year", "value", "raw", "annotation", "source_table"]]
    out = out.sort_values(["variable", "area_code", "year"]).reset_index(drop=True)

    manifest = []
    for _, s in spec.iterrows():
        v = out[(out["variable"] == s["name"]) & out["value"].notna()]
        years = sorted(v["year"].unique().tolist())
        manifest.append({
            "variable": s["name"], "code": s["code"], "name_ja": s["name_ja"], "source_table": s["sid"],
            "n_years": len(years), "first_year": years[0] if years else None,
            "last_year": years[-1] if years else None, "years": years,
            "areas_per_year_median": int(v.groupby("year")["area_code"].nunique().median()) if years else 0,
        })
    return out, manifest


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--level", choices=["pref", "muni"], action="append")
    ap.add_argument("--refresh", action="store_true")
    args = ap.parse_args()
    levels = args.level or ["pref", "muni"]

    RAW.mkdir(parents=True, exist_ok=True)
    mpath = RAW / "ssds_manifest.json"
    manifest = json.loads(mpath.read_text()) if mpath.exists() else {}

    for level in levels:
        path = RAW / f"ssds_{level}.parquet"
        if path.exists() and not args.refresh:
            print(f"[{level}] {path.name} exists, skipping (use --refresh)")
            continue
        df, man = download_level(level)
        df.to_parquet(path, index=False)
        manifest[level] = {"downloaded": date.today().isoformat(), "rows": len(df), "variables": man}
        print(f"[{level}] saved {len(df):,} rows -> {path.relative_to(estat.ROOT)}")
        for m in man:
            span = f"{m['first_year']}-{m['last_year']}" if m["n_years"] else "NO DATA"
            print(f"    {m['variable']:<26} {span:<10} {m['n_years']:>3} yrs  {m['areas_per_year_median']:>5} areas")

    mpath.write_text(json.dumps(manifest, ensure_ascii=False, indent=1))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile src/geo.py
"""Static prefecture attributes: English name, region, capital coordinates, distances.

Coordinates are approximate (prefectural capital city halls, 2 decimals). They are used
only to compute distance to the big metropolitan cores.
"""

from __future__ import annotations

import math

import pandas as pd

# code: (english name, region, capital lat, capital lon)
PREFS = {
    "01": ("Hokkaido", "Hokkaido", 43.06, 141.35),
    "02": ("Aomori", "Tohoku", 40.82, 140.74),
    "03": ("Iwate", "Tohoku", 39.70, 141.15),
    "04": ("Miyagi", "Tohoku", 38.27, 140.87),
    "05": ("Akita", "Tohoku", 39.72, 140.10),
    "06": ("Yamagata", "Tohoku", 38.24, 140.36),
    "07": ("Fukushima", "Tohoku", 37.75, 140.47),
    "08": ("Ibaraki", "Kanto", 36.34, 140.45),
    "09": ("Tochigi", "Kanto", 36.57, 139.88),
    "10": ("Gunma", "Kanto", 36.39, 139.06),
    "11": ("Saitama", "Kanto", 35.86, 139.65),
    "12": ("Chiba", "Kanto", 35.61, 140.12),
    "13": ("Tokyo", "Kanto", 35.69, 139.69),
    "14": ("Kanagawa", "Kanto", 35.45, 139.64),
    "15": ("Niigata", "Chubu", 37.90, 139.02),
    "16": ("Toyama", "Chubu", 36.70, 137.21),
    "17": ("Ishikawa", "Chubu", 36.59, 136.63),
    "18": ("Fukui", "Chubu", 36.07, 136.22),
    "19": ("Yamanashi", "Chubu", 35.66, 138.57),
    "20": ("Nagano", "Chubu", 36.65, 138.18),
    "21": ("Gifu", "Chubu", 35.39, 136.72),
    "22": ("Shizuoka", "Chubu", 34.98, 138.38),
    "23": ("Aichi", "Chubu", 35.18, 136.91),
    "24": ("Mie", "Kinki", 34.73, 136.51),
    "25": ("Shiga", "Kinki", 35.00, 135.87),
    "26": ("Kyoto", "Kinki", 35.02, 135.76),
    "27": ("Osaka", "Kinki", 34.69, 135.52),
    "28": ("Hyogo", "Kinki", 34.69, 135.18),
    "29": ("Nara", "Kinki", 34.69, 135.83),
    "30": ("Wakayama", "Kinki", 34.23, 135.17),
    "31": ("Tottori", "Chugoku", 35.50, 134.24),
    "32": ("Shimane", "Chugoku", 35.47, 133.05),
    "33": ("Okayama", "Chugoku", 34.66, 133.93),
    "34": ("Hiroshima", "Chugoku", 34.40, 132.46),
    "35": ("Yamaguchi", "Chugoku", 34.19, 131.47),
    "36": ("Tokushima", "Shikoku", 34.07, 134.56),
    "37": ("Kagawa", "Shikoku", 34.34, 134.04),
    "38": ("Ehime", "Shikoku", 33.84, 132.77),
    "39": ("Kochi", "Shikoku", 33.56, 133.53),
    "40": ("Fukuoka", "Kyushu-Okinawa", 33.61, 130.42),
    "41": ("Saga", "Kyushu-Okinawa", 33.25, 130.30),
    "42": ("Nagasaki", "Kyushu-Okinawa", 32.74, 129.87),
    "43": ("Kumamoto", "Kyushu-Okinawa", 32.79, 130.74),
    "44": ("Oita", "Kyushu-Okinawa", 33.24, 131.61),
    "45": ("Miyazaki", "Kyushu-Okinawa", 31.91, 131.42),
    "46": ("Kagoshima", "Kyushu-Okinawa", 31.56, 130.56),
    "47": ("Okinawa", "Kyushu-Okinawa", 26.21, 127.68),
}
TOKYO_AREA = {"11", "12", "13", "14"}  # Saitama, Chiba, Tokyo, Kanagawa
METROS = {"Tokyo": (35.69, 139.69), "Osaka": (34.69, 135.52), "Nagoya": (35.18, 136.91)}


def haversine_km(a: tuple[float, float], b: tuple[float, float]) -> float:
    lat1, lon1, lat2, lon2 = map(math.radians, (*a, *b))
    h = math.sin((lat2 - lat1) / 2) ** 2 + math.cos(lat1) * math.cos(lat2) * math.sin((lon2 - lon1) / 2) ** 2
    return 2 * 6371 * math.asin(math.sqrt(h))


def pref_table() -> pd.DataFrame:
    rows = []
    for code, (name, region, lat, lon) in PREFS.items():
        d = {m: haversine_km((lat, lon), xy) for m, xy in METROS.items()}
        rows.append({
            "pref": code, "pref_name": name, "region": region,
            "dist_tokyo_km": round(d["Tokyo"], 1),
            "dist_metro_km": round(min(d.values()), 1),  # nearest of Tokyo / Osaka / Nagoya
        })
    return pd.DataFrame(rows)

In [ ]:
%%writefile src/build_panel.py
"""Step 5: clean the raw SSDS downloads and build 5-year panels.

Usage:
    python src/build_panel.py

Outputs (data/processed/):
    pref_yearly.parquet  wide table, one row per prefecture (and national) x year, derived rates
    pref_panel.parquet   one row per prefecture x 5-year period (1980->85 ... 2015->20)
    muni_panel.parquet   one row per municipality x 5-year period (1995->2000 ... 2015->20)
    pref_start_2020.parquet, muni_start_2020.parquet  features at 2020, for forecasting

Conventions (see docs/datasets.md for the reasons):
- Period t->t+5 runs from the 1 Oct t population count to the 1 Oct t+5 count. Births and
  deaths are summed over years t+1..t+5, which matched the population change best.
- Targets, as % of the population at t over the whole 5 years:
    growth  = (P[t+5] - P[t]) / P[t]
    natural = (births - deaths) / P[t]
    social  = growth - natural  (residual net migration, includes foreign residents)
- Features are measured at the start of the period (year t). Variables only collected in
  survey years (census, housing survey) take the latest value from years <= t, at most
  `MAX_LAG` years old.
- D1: Fukushima is dropped from periods starting 2010 or later.
- D5: designated-city wards are dropped (the city itself is kept); Tokyo's 23 wards are kept.
"""

from __future__ import annotations

import numpy as np
import pandas as pd

import geo
from estat import ROOT

RAW = ROOT / "data" / "raw"
OUT = ROOT / "data" / "processed"
MAX_LAG = 5
NATIONAL = "00000"
FUKUSHIMA = "07"


def wide(path) -> pd.DataFrame:
    df = pd.read_parquet(path)
    names = df.drop_duplicates("area_code").set_index("area_code")[
        [c for c in ("area_name", "area_level") if c in df]]
    w = df.pivot_table(index=["area_code", "year"], columns="variable", values="value").reset_index()
    w.columns.name = None
    return w.merge(names, left_on="area_code", right_index=True)


def asof(w: pd.DataFrame, col: str, max_lag: int = MAX_LAG) -> pd.Series:
    """Latest non-missing value of `col` from the same or earlier years (within max_lag)."""
    s = w[["area_code", "year", col]].copy()
    s["_src"] = s["year"].where(s[col].notna())
    s[["_v", "_src"]] = s.groupby("area_code")[[col, "_src"]].ffill()
    return s["_v"].where(s["year"] - s["_src"] <= max_lag)


def period_targets(w: pd.DataFrame, starts: list[int]) -> pd.DataFrame:
    """5-year growth / natural / social change for each area and period start t."""
    idx = w.set_index(["area_code", "year"])
    pop, births, deaths = idx["population"], idx["births"], idx["deaths"]
    rows = []
    for t in starts:
        p0 = pop.xs(t, level="year")
        years = pop.index.get_level_values("year")
        p1 = pop.xs(t + 5, level="year") if (t + 5) in years else pd.Series(np.nan, index=p0.index)
        yrs = range(t + 1, t + 6)
        sel = births.index.get_level_values("year").isin(yrs)
        b = births[sel].groupby(level="area_code").agg(lambda s: s.sum(min_count=5))
        d = deaths[sel].groupby(level="area_code").agg(lambda s: s.sum(min_count=5))
        r = pd.DataFrame({"pop_start": p0, "pop_end": p1, "births_5y": b, "deaths_5y": d})
        r["period"] = t
        rows.append(r)
    out = pd.concat(rows).reset_index().rename(columns={"index": "area_code"})
    out["growth"] = (out.pop_end - out.pop_start) / out.pop_start * 100
    out["natural"] = (out.births_5y - out.deaths_5y) / out.pop_start * 100
    out["social"] = out.growth - out.natural
    return out


# --------------------------------------------------------------------------- prefecture

def build_pref() -> tuple[pd.DataFrame, pd.DataFrame]:
    w = wide(RAW / "ssds_pref.parquet").sort_values(["area_code", "year"]).reset_index(drop=True)
    w["pref"] = w.area_code.str[:2]

    # Census-only / survey-year counts and rates -> carry forward to non-survey years.
    for col in ["unmarried_f_30_34", "did_population", "unemployment_rate", "vacancy_rate", "doctors"]:
        w[col] = asof(w, col)
    # All period starts (1980, 1985, ...) are census years, so census-only counts such as
    # unmarried_f_30_34 line up with the yearly female_30_34 denominator at those years.

    w["aging_rate"] = w.pop_65plus / w.population * 100
    w["young_women_share"] = w[["female_20_24", "female_25_29", "female_30_34", "female_35_39"]].sum(
        axis=1, min_count=4) / w.population * 100
    w["unmarried_f_30_34_rate"] = w.unmarried_f_30_34 / w.female_30_34 * 100
    w["did_share"] = w.did_population / w.population * 100
    w["doctors_per_100k"] = w.doctors / w.population * 1e5
    w["log_population"] = np.log(w.population.where(w.population > 0))
    w["income_per_taxpayer"] = w.taxable_income / w.taxpayers  # thousand yen
    w["univ_outflow_share"] = w.univ_entrants_other_pref / w.univ_entrants_total * 100

    # Nominal money values -> relative to the national level in the same year (removes inflation).
    nat = w[w.area_code == NATIONAL].set_index("year")["income_per_taxpayer"]
    w["income_rel"] = w.income_per_taxpayer / w.year.map(nat) * 100
    lp_mean = w[w.area_code != NATIONAL].groupby("year")["land_price_res"].mean()
    w["land_price_rel"] = w.land_price_res / w.year.map(lp_mean) * 100

    # Snow is noisy year to year: 5-year trailing mean.
    w["snow_days_5y"] = w.groupby("area_code")["snow_days"].transform(lambda s: s.rolling(5, min_periods=3).mean())

    yearly = w.merge(geo.pref_table(), on="pref", how="left")

    feats = ["aging_rate", "young_women_share", "tfr", "unmarried_f_30_34_rate", "income_rel",
             "job_openings_ratio", "unemployment_rate", "did_share", "vacancy_rate", "land_price_rel",
             "univ_outflow_share", "hs_grads_jobs_other_pref", "univ_capacity_index", "local_univ_entry_share",
             "doctors_per_100k", "fiscal_strength", "snow_days_5y", "log_population"]
    pw = w[w.area_code != NATIONAL]
    tg = period_targets(pw, list(range(1980, 2025, 5)))  # 2020 start = forecast rows (no target yet)
    start = pw[["area_code", "year", *feats]].rename(columns={"year": "period"})
    panel = tg.merge(start, on=["area_code", "period"], how="left")
    panel["pref"] = panel.area_code.str[:2]
    panel = panel.merge(geo.pref_table(), on="pref", how="left")

    panel["disaster_2011"] = ((panel.period == 2010) & panel.pref.isin(["03", "04"])).astype(int)  # Iwate, Miyagi
    panel["univ_data_available"] = panel.univ_outflow_share.notna().astype(int)
    panel = panel[~((panel.pref == FUKUSHIMA) & (panel.period >= 2010))]  # D1
    panel["period_label"] = panel.period.astype(str) + "-" + (panel.period + 5).astype(str)
    panel = panel.reset_index(drop=True)
    return yearly, panel[panel.period < 2020].reset_index(drop=True), panel[panel.period == 2020].reset_index(drop=True)


# --------------------------------------------------------------------------- municipality

def build_muni(pref_yearly: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    w = wide(RAW / "ssds_muni.parquet").sort_values(["area_code", "year"]).reset_index(drop=True)

    # D5: keep cities/towns/villages; drop designated-city wards and the all-Tokyo-wards
    # aggregate (13100), keep the 23 Tokyo special wards.
    tokyo_wards = w.area_code.between("13101", "13123")
    keep = ((w.area_level == "2") & (w.area_code != "13100")) | tokyo_wards
    w = w[keep].copy()
    w["pref"] = w.area_code.str[:2]

    # Counts reported as '-' (no DID in this municipality) mean zero, not missing.
    w["did_population"] = w.did_population.where(w.population.isna() | w.did_population.notna(), 0)
    for col in ["vacancy_rate", "unemployment_rate", "did_population", "female_15_64", "pop_65plus", "doctors"]:
        w[col] = asof(w, col)

    w["aging_rate"] = w.pop_65plus / w.population * 100
    w["working_women_share"] = w.female_15_64 / w.population * 100
    w["did_share"] = w.did_population / w.population * 100
    w["doctors_per_1k"] = w.doctors / w.population * 1e3
    w["log_population"] = np.log(w.population.where(w.population > 0))
    nat = pref_yearly[pref_yearly.area_code == NATIONAL].set_index("year")["income_per_taxpayer"]
    w["income_rel"] = (w.taxable_income / w.taxpayers) / w.year.map(nat) * 100

    feats = ["aging_rate", "working_women_share", "income_rel", "unemployment_rate", "did_share",
             "vacancy_rate", "doctors_per_1k", "fiscal_strength", "log_population"]
    tg = period_targets(w, list(range(1995, 2025, 5)))  # 2020 start = forecast rows
    start = w[["area_code", "year", "area_name", *feats]].rename(columns={"year": "period"})
    panel = tg.merge(start, on=["area_code", "period"], how="left")
    panel["pref"] = panel.area_code.str[:2]
    panel = panel.merge(geo.pref_table()[["pref", "pref_name", "region", "dist_tokyo_km", "dist_metro_km"]],
                        on="pref", how="left")
    panel = panel[~((panel.pref == FUKUSHIMA) & (panel.period >= 2010))]  # D1
    # Same logic as D1: Miyake village (13381) was fully evacuated after the 2000 eruption
    # (census 2000 population = 0), so drop the two periods touching 2000.
    panel = panel[~((panel.area_code == "13381") & panel.period.isin([1995, 2000]))]
    panel["period_label"] = panel.period.astype(str) + "-" + (panel.period + 5).astype(str)
    future = panel[(panel.period == 2020) & (panel.pop_start > 0)].reset_index(drop=True)
    panel = panel[panel.period < 2020].dropna(subset=["growth", "natural"])
    return panel.reset_index(drop=True), future


def main() -> None:
    OUT.mkdir(parents=True, exist_ok=True)
    yearly, pref, pref_2020 = build_pref()
    muni, muni_2020 = build_muni(yearly)
    yearly.to_parquet(OUT / "pref_yearly.parquet", index=False)
    pref.to_parquet(OUT / "pref_panel.parquet", index=False)
    muni.to_parquet(OUT / "muni_panel.parquet", index=False)
    # Start-of-period features for 2020 (targets unknown) -> used by the forecast in step 8.
    pref_2020.to_parquet(OUT / "pref_start_2020.parquet", index=False)
    muni_2020.to_parquet(OUT / "muni_start_2020.parquet", index=False)

    for name, df in [("pref_panel", pref), ("muni_panel", muni)]:
        print(f"\n{name}: {len(df):,} rows, {df.area_code.nunique()} areas, periods {sorted(df.period.unique())}")
        miss = df.isna().mean().mul(100).round(1)
        print("  % missing:", miss[miss > 0].to_dict())


if __name__ == "__main__":
    main()

In [ ]:
%%writefile src/plotstyle.py
"""Shared matplotlib style (reference palette from the dataviz skill, light mode).

Categorical slots are assigned in fixed order. Only the first three are used in charts
where every series can sit next to every other (scatter, multi-line), because only those
three stay distinguishable for colour-blind readers in all pairings.
"""

from __future__ import annotations

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

from estat import ROOT

FIG_DIR = ROOT / "reports" / "figures"

SURFACE = "#fcfcfb"
INK = "#0b0b0b"
INK_2 = "#52514e"
MUTED = "#898781"
GRID = "#e1e0d9"
BASELINE = "#c3c2b7"
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
NEUTRAL_MID = "#f0efec"
BLUE_RAMP = ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"]
RED_RAMP = ["#fbdada", "#f4a9a9", "#ec7a7a", "#e34948", "#c23736", "#9a2a29", "#701d1d"]

# Diverging: red = decline, gray = no change, blue = growth.
DIVERGING = LinearSegmentedColormap.from_list("decline_growth", RED_RAMP[::-1][1:] + [NEUTRAL_MID] + BLUE_RAMP[1:])


def apply() -> None:
    mpl.rcParams.update({
        "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
        "font.family": "sans-serif", "font.size": 10,
        "font.sans-serif": ["Noto Sans CJK JP", "IPAexGothic", "DejaVu Sans"],
        "text.color": INK, "axes.labelcolor": INK_2, "xtick.color": MUTED, "ytick.color": MUTED,
        "axes.edgecolor": BASELINE, "axes.spines.top": False, "axes.spines.right": False,
        "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6, "axes.axisbelow": True,
        "axes.titlesize": 12, "axes.titleweight": "bold", "axes.titlelocation": "left",
        "lines.linewidth": 2, "lines.markersize": 4, "legend.frameon": False,
        "figure.dpi": 110, "savefig.dpi": 160, "savefig.bbox": "tight",
    })


def save(fig, name: str) -> None:
    FIG_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIG_DIR / f"{name}.png")
    plt.close(fig)
    print(f"  saved reports/figures/{name}.png")

In [ ]:
%%writefile src/eda.py
"""Step 6: exploratory analysis.

Usage:
    python src/eda.py

Writes figures to reports/figures/ and key numbers to reports/eda_summary.json.
Questions answered:
  - How did Japan's population, and its natural vs. social components, evolve?
  - When did each prefecture's population peak (start of sustained decline), and does
    that line up with historical events?
  - How does migration toward the Tokyo area track the economy (bubble, crises, COVID)?
  - Where and how fast are municipalities shrinking?
"""

from __future__ import annotations

import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import geo
import plotstyle as ps
from estat import ROOT

PROC = ROOT / "data" / "processed"
EVENTS = [(1991, "Bubble bursts"), (2008, "Lehman shock"), (2011, "Tohoku quake"), (2020, "COVID-19")]


def mark_events(ax, ymax_frac=0.97):
    lo, hi = ax.get_ylim()
    for i, (yr, label) in enumerate(EVENTS):
        ax.axvline(yr, color=ps.BASELINE, lw=1, ls=(0, (3, 3)), zorder=0)
        frac = ymax_frac - 0.07 * (i % 2)  # stagger so neighbouring events don't collide
        ax.text(yr + 0.3, lo + (hi - lo) * frac, label, color=ps.MUTED, fontsize=8, va="top")


def national(y: pd.DataFrame, summary: dict) -> None:
    n = y[y.area_code == "00000"].set_index("year")
    peak_year = int(n.population.idxmax())
    summary["national_peak_year"] = peak_year
    summary["national_peak_population"] = int(n.population.max())

    fig, ax = plt.subplots(figsize=(8, 3.6))
    ax.plot(n.index, n.population / 1e6, color=ps.SERIES[0])
    ax.scatter([peak_year], [n.population.max() / 1e6], color=ps.SERIES[0], s=40, zorder=3,
               edgecolor=ps.SURFACE, linewidth=2)
    ax.annotate(f"Peak {peak_year}: {n.population.max() / 1e6:.1f}M", (peak_year, n.population.max() / 1e6),
                xytext=(-10, 10), textcoords="offset points", ha="right", color=ps.INK_2, fontsize=9)
    ax.set_title("Japan's population peaked in the late 2000s")
    ax.set_ylabel("Population (millions)")
    ax.set_xlim(1975, 2025)
    ps.save(fig, "01_national_population")

    nat = (n.births - n.deaths) / 1e3
    fig, ax = plt.subplots(figsize=(8, 3.6))
    ax.axhline(0, color=ps.BASELINE, lw=1)
    ax.plot(nat.dropna().index, nat.dropna(), color=ps.SERIES[1])
    first_neg = int(nat[nat < 0].index.min())
    summary["national_first_natural_decrease_year"] = first_neg
    ax.annotate(f"Deaths exceed births from {first_neg}", (first_neg, 0), xytext=(-12, -30),
                textcoords="offset points", ha="right", color=ps.INK_2, fontsize=9,
                arrowprops=dict(arrowstyle="-", color=ps.MUTED, lw=0.8))
    ax.set_xlim(1975, 2025)
    ax.set_title("Natural change: births minus deaths, Japan")
    ax.set_ylabel("Thousand people per year")
    mark_events(ax)
    ps.save(fig, "02_national_natural_change")


def peaks(y: pd.DataFrame, summary: dict) -> pd.DataFrame:
    p = y[y.area_code != "00000"].dropna(subset=["population"])
    rows = []
    for code, g in p.groupby("area_code"):
        g = g.set_index("year").population
        peak = int(g.idxmax())
        rows.append({"pref": code[:2], "peak_year": peak, "censored": peak == g.index.min(),
                     "still_growing": peak == g.index.max(),
                     "change_since_peak_pct": (g.iloc[-1] / g.max() - 1) * 100})
    pk = pd.DataFrame(rows).merge(geo.pref_table(), on="pref").sort_values("peak_year")

    bins = [(1975, 1975, "Already declining in 1975"), (1976, 1985, "1976–1985"),
            (1986, 1991, "Bubble era 1986–1991"), (1992, 1999, "After the bubble 1992–1999"),
            (2000, 2004, "2000–2004"), (2005, 2010, "2005–2010 (national peak)"),
            (2011, 2019, "2011–2019"), (2020, 2025, "2020 or later / still growing")]
    summary["peak_year_groups"] = {lab: pk[(pk.peak_year >= a) & (pk.peak_year <= b)].pref_name.tolist()
                                   for a, b, lab in bins}

    fig, ax = plt.subplots(figsize=(8, 9.5))
    ypos = np.arange(len(pk))
    ax.hlines(ypos, 1975, pk.peak_year, color=ps.GRID, lw=1)
    ax.scatter(pk.peak_year, ypos, s=36, color=ps.SERIES[0], zorder=3, edgecolor=ps.SURFACE, linewidth=1.5)
    for i, r in enumerate(pk.itertuples()):
        tag = " (≤1975)" if r.censored else (" (still growing)" if r.still_growing else "")
        ax.text(r.peak_year + 0.6, i, f"{r.pref_name}{tag}", va="center", fontsize=8, color=ps.INK_2)
    ax.set_yticks([])
    ax.grid(axis="y", visible=False)
    ax.set_xlim(1974, 2034)
    ax.set_ylim(-1, len(pk))
    ax.invert_yaxis()
    for i, (yr, label) in enumerate(EVENTS):
        ax.axvline(yr, color=ps.BASELINE, lw=1, ls=(0, (3, 3)), zorder=0)
        ax.text(yr, -1.4 - 1.1 * (i % 2), label, color=ps.MUTED, fontsize=8, ha="center")
    ax.set_title("Year each prefecture's population peaked (1975–2024 data)", pad=34)
    ax.set_xlabel("Peak year")
    ps.save(fig, "03_prefecture_peak_year")
    pk.to_csv(ROOT / "reports" / "prefecture_peaks.csv", index=False)
    return pk


def migration_blocs(y: pd.DataFrame, summary: dict) -> None:
    p = y[y.area_code != "00000"].copy()
    bloc = {**{c: "Tokyo area" for c in geo.TOKYO_AREA},
            **{c: "Osaka & Nagoya areas" for c in ["21", "23", "24", "26", "27", "28", "29"]}}
    p["bloc"] = p.pref.map(bloc).fillna("Rest of Japan")
    g = p.groupby(["bloc", "year"])[["moves_in", "moves_out", "population"]].sum(min_count=1)
    g["net_rate"] = (g.moves_in - g.moves_out) / g.population * 1000
    order = ["Tokyo area", "Osaka & Nagoya areas", "Rest of Japan"]

    fig, ax = plt.subplots(figsize=(8.5, 4))
    ax.axhline(0, color=ps.BASELINE, lw=1)
    for i, b in enumerate(order):
        s = g.loc[b].net_rate.dropna()
        s = s[s.index <= 2024]
        ax.plot(s.index, s, color=ps.SERIES[i])
        ax.text(s.index[-1] + 0.6, s.iloc[-1], b, color=ps.INK_2, fontsize=9, va="center")
    ax.set_xlim(1975, 2035)
    ax.set_title("Net migration between prefectures, by region group")
    ax.set_ylabel("Net moves in per 1,000 residents")
    mark_events(ax)
    ps.save(fig, "04_migration_blocs")

    tk = g.loc["Tokyo area"].net_rate
    summary["tokyo_area_net_migration_per_1000"] = {int(k): round(v, 2) for k, v in tk.dropna().items()
                                                   if k in (1987, 1994, 2007, 2019, 2021, 2024)}
    summary["tokyo_area_min_year_1985_2000"] = int(tk.loc[1985:2000].idxmin())


def heatmap(panel: pd.DataFrame, summary: dict) -> None:
    h = panel.pivot_table(index="pref_name", columns="period_label", values="growth")
    h = h.loc[h.mean(axis=1).sort_values().index]
    summary["prefs_declining_by_period"] = (panel.assign(dec=panel.growth < 0)
                                            .groupby("period_label").dec.sum().astype(int).to_dict())
    lim = np.nanmax(np.abs(h.values))
    fig, ax = plt.subplots(figsize=(7, 11))
    im = ax.imshow(h.values, cmap=ps.DIVERGING, vmin=-lim, vmax=lim, aspect="auto")
    ax.set_yticks(range(len(h)), h.index, fontsize=8)
    ax.set_xticks(range(h.shape[1]), h.columns, rotation=45, ha="right", fontsize=8)
    ax.grid(False)
    for s in ax.spines.values():
        s.set_visible(False)
    ax.set_xticks(np.arange(-.5, h.shape[1]), minor=True)
    ax.set_yticks(np.arange(-.5, len(h)), minor=True)
    ax.grid(which="minor", color=ps.SURFACE, lw=2)
    ax.tick_params(which="minor", length=0)
    cb = fig.colorbar(im, ax=ax, shrink=0.4, pad=0.02)
    cb.set_label("Population change over 5 years (%)", color=ps.INK_2)
    cb.outline.set_visible(False)
    ax.set_title("Population change by prefecture and period\n(red = decline, blue = growth; Fukushima 2010+ excluded)")
    ps.save(fig, "05_prefecture_growth_heatmap")


def decomposition(panel: pd.DataFrame) -> None:
    periods = sorted(panel.period_label.unique())
    fig, axes = plt.subplots(2, 4, figsize=(11, 5.6), sharex=True, sharey=True)
    lim = np.ceil(max(panel.natural.abs().max(), panel.social.abs().max()))
    for ax, per in zip(axes.flat, periods):
        d = panel[panel.period_label == per]
        ax.axhline(0, color=ps.BASELINE, lw=1)
        ax.axvline(0, color=ps.BASELINE, lw=1)
        ax.plot([-lim, lim], [lim, -lim], color=ps.GRID, lw=1, ls=(0, (2, 2)))
        ax.scatter(d.natural, d.social, s=16, color=ps.SERIES[0], edgecolor=ps.SURFACE, linewidth=1)
        for name in ("Tokyo", "Akita", "Okinawa"):
            r = d[d.pref_name == name]
            if len(r):
                ax.annotate(name, (r.natural.iloc[0], r.social.iloc[0]), xytext=(4, 3),
                            textcoords="offset points", fontsize=7, color=ps.INK_2)
        ax.set_title(per, fontsize=10, fontweight="normal")
        ax.set_xlim(-lim, lim)
        ax.set_ylim(-lim, lim)
    fig.supxlabel("Natural change (% over 5 years)", color=ps.INK_2, fontsize=10)
    fig.supylabel("Social change / net migration (%)", color=ps.INK_2, fontsize=10)
    fig.suptitle("Each prefecture's natural vs. social change; below the dashed line = shrinking",
                 x=0.01, ha="left", fontweight="bold")
    fig.tight_layout()
    ps.save(fig, "06_natural_vs_social")


def municipalities(m: pd.DataFrame, summary: dict) -> None:
    s = m.groupby("period_label").agg(
        shrinking=("growth", lambda x: (x < 0).mean() * 100),
        fast=("growth", lambda x: (x < -10).mean() * 100),
        nat_neg=("natural", lambda x: (x < 0).mean() * 100),
        soc_neg=("social", lambda x: (x < 0).mean() * 100))
    summary["municipal_shares_pct"] = s.round(1).to_dict(orient="index")

    fig, ax = plt.subplots(figsize=(8, 3.8))
    cols = [("shrinking", "Population shrinking"), ("nat_neg", "More deaths than births"),
            ("fast", "Shrinking more than 10%")]
    for i, (c, lab) in enumerate(cols):
        ax.plot(range(len(s)), s[c], color=ps.SERIES[i], marker="o", markersize=8,
                markeredgecolor=ps.SURFACE, markeredgewidth=2)
        ax.text(len(s) - 1 + 0.12, s[c].iloc[-1], f"{lab} ({s[c].iloc[-1]:.0f}%)", va="center",
                fontsize=9, color=ps.INK_2)
    ax.set_xticks(range(len(s)), s.index)
    ax.set_xlim(-0.2, len(s) + 1.8)
    ax.set_ylim(0, 100)
    ax.set_ylabel("% of municipalities")
    ax.set_title("Share of the 1,741 municipalities that are shrinking")
    ps.save(fig, "07_municipal_shrinking_share")

    # Size matters: small municipalities shrink much faster.
    m = m.copy()
    m["size"] = pd.cut(m.pop_start, [0, 5e3, 2e4, 5e4, 2e5, np.inf],
                       labels=["<5k", "5–20k", "20–50k", "50–200k", "200k+"])
    t = m[m.period == 2015].groupby("size", observed=True)[["growth", "natural", "social"]].median()
    summary["municipal_median_2015_by_size"] = t.round(2).to_dict(orient="index")
    fig, ax = plt.subplots(figsize=(7, 3.8))
    x = np.arange(len(t))
    w = 0.38
    for i, (c, lab) in enumerate([("natural", "Natural change"), ("social", "Social change")]):
        bars = ax.bar(x + (i - 0.5) * (w + 0.02), t[c], width=w, color=ps.SERIES[i], label=lab)
        for b, v in zip(bars, t[c]):
            ax.text(b.get_x() + b.get_width() / 2, v - 0.25 if v < 0 else v + 0.1, f"{v:.1f}",
                    ha="center", va="top" if v < 0 else "bottom", fontsize=8, color=ps.INK_2)
    ax.axhline(0, color=ps.BASELINE, lw=1)
    ax.set_xticks(x, t.index)
    ax.set_xlabel("Municipality population in 2015")
    ax.set_ylabel("Median change 2015–2020 (%)")
    ax.legend(loc="lower right")
    ax.set_title("Small municipalities lose people both ways")
    ps.save(fig, "08_municipal_change_by_size")


def correlations(panel: pd.DataFrame, summary: dict) -> None:
    feats = ["aging_rate", "young_women_share", "tfr", "unmarried_f_30_34_rate", "income_rel",
             "job_openings_ratio", "unemployment_rate", "did_share", "vacancy_rate", "land_price_rel",
             "univ_outflow_share", "hs_grads_jobs_other_pref", "univ_capacity_index", "doctors_per_100k",
             "fiscal_strength", "snow_days_5y", "log_population", "dist_metro_km"]
    c = panel[feats + ["natural", "social"]].corr(method="spearman").loc[feats, ["natural", "social"]]
    summary["spearman_with_targets"] = c.round(2).to_dict(orient="index")
    c = c.sort_values("social")
    fig, axes = plt.subplots(1, 2, figsize=(10, 6), sharey=True)
    for ax, col, color in zip(axes, ["natural", "social"], ps.SERIES[:2]):
        ax.barh(range(len(c)), c[col], color=color, height=0.7)
        ax.axvline(0, color=ps.BASELINE, lw=1)
        ax.set_xlim(-1, 1)
        ax.set_title(f"vs. {col} change", fontsize=10)
        ax.grid(axis="y", visible=False)
        for i, v in enumerate(c[col]):
            ax.text(v + (0.03 if v >= 0 else -0.03), i, f"{v:.2f}", va="center",
                    ha="left" if v >= 0 else "right", fontsize=7, color=ps.INK_2)
    axes[0].set_yticks(range(len(c)), c.index, fontsize=8)
    fig.suptitle("Rank correlation of start-of-period features with the next 5 years' change (prefectures)",
                 x=0.01, ha="left", fontweight="bold")
    fig.tight_layout()
    ps.save(fig, "09_feature_correlations")


def main() -> None:
    ps.apply()
    (ROOT / "reports").mkdir(exist_ok=True)
    y = pd.read_parquet(PROC / "pref_yearly.parquet")
    pref = pd.read_parquet(PROC / "pref_panel.parquet")
    muni = pd.read_parquet(PROC / "muni_panel.parquet")
    summary: dict = {}
    national(y, summary)
    peaks(y, summary)
    migration_blocs(y, summary)
    heatmap(pref, summary)
    decomposition(pref)
    municipalities(muni, summary)
    correlations(pref, summary)
    (ROOT / "reports" / "eda_summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=1))
    print(json.dumps(summary, ensure_ascii=False, indent=1)[:6000])


if __name__ == "__main__":
    main()

In [ ]:
%%writefile src/features.py
"""Feature lists and transforms shared by steps 7-9."""

from __future__ import annotations

import numpy as np
import pandas as pd

from estat import ROOT

PROC = ROOT / "data" / "processed"
TARGETS = ["natural", "social"]

# Prefecture candidates. univ_outflow_share only exists from 2000 (D4), so it is in a
# separate list for the post-2000 models.
PREF_FEATURES = [
    "aging_rate", "young_women_share", "tfr", "unmarried_f_30_34_rate",           # demography
    "income_rel", "job_openings_ratio", "unemployment_rate",                      # economy
    "did_share", "log_dist_metro", "log_population",                               # urban / geography
    "vacancy_rate", "log_land_price_rel",                                          # housing
    "hs_grads_jobs_other_pref", "univ_capacity_index",                             # education
    "doctors_per_100k", "fiscal_strength", "snow_days_5y",                         # health, finance, climate
]
PREF_FEATURES_POST2000 = PREF_FEATURES + ["univ_outflow_share"]

MUNI_FEATURES = ["aging_rate", "income_rel", "unemployment_rate", "did_share", "log_population",
                 "doctors_per_1k", "fiscal_strength", "log_dist_metro"]
MUNI_FEATURES_EXTRA = MUNI_FEATURES + ["working_women_share", "vacancy_rate"]  # shorter coverage

LABELS = {
    "aging_rate": "Aging rate (65+ %)", "young_women_share": "Women 20–39 (% of pop.)",
    "working_women_share": "Women 15–64 (% of pop.)", "tfr": "Total fertility rate",
    "unmarried_f_30_34_rate": "Unmarried women 30–34 (%)", "income_rel": "Income per taxpayer (Japan=100)",
    "job_openings_ratio": "Job-openings ratio", "unemployment_rate": "Unemployment rate",
    "did_share": "Urban share (DID %)", "log_dist_metro": "Distance to nearest metro (log)",
    "log_population": "Population size (log)", "vacancy_rate": "Vacant housing (%)",
    "log_land_price_rel": "Land price (log, rel.)", "hs_grads_jobs_other_pref": "HS grads taking jobs elsewhere (%)",
    "univ_capacity_index": "University capacity index", "univ_outflow_share": "Univ. entrants leaving pref. (%)",
    "doctors_per_100k": "Doctors per 100k", "doctors_per_1k": "Doctors per 1k",
    "fiscal_strength": "Fiscal strength index", "snow_days_5y": "Snow days per year",
    "disaster_2011": "2011 disaster (Iwate, Miyagi)",
}


def add_transforms(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["log_dist_metro"] = np.log1p(df["dist_metro_km"])
    if "land_price_rel" in df:
        df["log_land_price_rel"] = np.log(df["land_price_rel"])
    return df


def load_pref() -> pd.DataFrame:
    return add_transforms(pd.read_parquet(PROC / "pref_panel.parquet"))


def load_muni() -> pd.DataFrame:
    return add_transforms(pd.read_parquet(PROC / "muni_panel.parquet"))


def demean_by(df: pd.DataFrame, cols: list[str], by: str = "period") -> pd.DataFrame:
    """Subtract the group mean (e.g. period fixed effect) from each column."""
    return df[cols] - df.groupby(by)[cols].transform("mean")

In [ ]:
%%writefile src/select_features.py
"""Step 7: attribute (feature) selection.

Usage:
    python src/select_features.py

For each level (prefecture, municipality) and target (natural, social change):
  1. Period-demean features and target, so selection compares areas *within* the same
     period and isn't driven by the shared national trend.
  2. VIF on the standardized candidates (how much each one overlaps with the others).
  3. LassoCV with grouped CV (folds = whole areas, so an area never sits in both train and test).
  4. Stability selection: Lasso at the CV-chosen alpha on bootstrap resamples of whole areas;
     selection frequency per feature.
  5. Random-forest permutation importance, out-of-fold.
Then a rule picks the final set: stability >= 0.6, then among pairs correlated |r| > 0.8 keep
the more stable one, capped at 8 features.

Outputs: reports/feature_selection.md, reports/feature_selection_<level>.csv,
config/selected_features.json, figure 10.
"""

from __future__ import annotations

import inspect
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.linear_model import Lasso, LassoCV
from sklearn.model_selection import GroupKFold
from statsmodels.stats.outliers_influence import variance_inflation_factor

import features as F
import plotstyle as ps
from estat import ROOT

# scikit-learn >= 1.7 takes the number of alphas as `alphas=int` (n_alphas was removed in 1.9);
# older versions (e.g. on Google Colab) only accept `n_alphas`.
N_ALPHAS = {"n_alphas": 60} if "n_alphas" in inspect.signature(LassoCV).parameters else {"alphas": 60}
RNG = np.random.default_rng(42)
N_BOOT = 300
STABILITY_MIN = 0.6
CORR_MAX = 0.8
MAX_FEATURES = 8


def prep(df: pd.DataFrame, feats: list[str], target: str):
    d = df.dropna(subset=feats + [target]).reset_index(drop=True)
    X = F.demean_by(d, feats)
    y = F.demean_by(d, [target])[target]
    X = (X - X.mean()) / X.std()
    return d, X, y


def vif(X: pd.DataFrame) -> pd.Series:
    return pd.Series([variance_inflation_factor(X.values, i) for i in range(X.shape[1])], index=X.columns)


def run(df: pd.DataFrame, feats: list[str], target: str, level: str, n_splits: int = 5) -> pd.DataFrame:
    d, X, y = prep(df, feats, target)
    groups = d["area_code"].values
    cv = GroupKFold(n_splits=n_splits)

    lcv = LassoCV(cv=list(cv.split(X, y, groups)), max_iter=20000, **N_ALPHAS).fit(X, y)
    alpha = lcv.alpha_
    lasso_r2 = 1 - lcv.mse_path_.mean(axis=1).min() / y.var()

    areas = np.unique(groups)
    pos = {a: np.where(groups == a)[0] for a in areas}
    freq = np.zeros(len(feats))
    for _ in range(N_BOOT):
        idx = np.concatenate([pos[a] for a in RNG.choice(areas, size=len(areas), replace=True)])
        coef = Lasso(alpha=alpha, max_iter=20000).fit(X.iloc[idx], y.iloc[idx]).coef_
        freq += coef != 0
    freq /= N_BOOT

    imp = np.zeros(len(feats))
    oof_ss = 0.0
    for tr, te in cv.split(X, y, groups):
        rf = RandomForestRegressor(n_estimators=300, min_samples_leaf=5, n_jobs=-1, random_state=0)
        rf.fit(X.iloc[tr], y.iloc[tr])
        oof_ss += ((y.iloc[te] - rf.predict(X.iloc[te])) ** 2).sum()
        pi = permutation_importance(rf, X.iloc[te], y.iloc[te], n_repeats=5, random_state=0, n_jobs=-1)
        imp += pi.importances_mean / n_splits
    rf_r2 = 1 - oof_ss / ((y - y.mean()) ** 2).sum()

    res = pd.DataFrame({
        "feature": feats, "label": [F.LABELS.get(f, f) for f in feats],
        "lasso_coef": lcv.coef_, "stability": freq, "rf_importance": imp, "vif": vif(X).values,
        "spearman": [X[f].corr(y, method="spearman") for f in feats],
    })
    res["level"], res["target"], res["n_obs"] = level, target, len(d)
    res["lasso_cv_r2"], res["rf_cv_r2"] = lasso_r2, rf_r2
    res["selected"] = False
    # Selection rule.
    corr = X.corr().abs()
    chosen: list[str] = []
    for f in res.sort_values(["stability", "rf_importance"], ascending=False).feature:
        s = res.set_index("feature").loc[f, "stability"]
        if s < STABILITY_MIN or len(chosen) >= MAX_FEATURES:
            continue
        if all(corr.loc[f, c] <= CORR_MAX for c in chosen):
            chosen.append(f)
    res.loc[res.feature.isin(chosen), "selected"] = True
    print(f"[{level}/{target}] n={len(d)}  lasso CV R2={lasso_r2:.2f}  RF CV R2={rf_r2:.2f}  selected={chosen}")
    return res.sort_values("stability", ascending=False)


def figure(results: list[pd.DataFrame], name: str, title: str) -> None:
    fig, axes = plt.subplots(1, len(results), figsize=(5.2 * len(results), 0.32 * len(results[0]) + 1.6),
                             sharey=False)
    for ax, r in zip(np.atleast_1d(axes), results):
        r = r.sort_values("stability")
        colors = [ps.SERIES[0] if s else ps.BLUE_RAMP[1] for s in r.selected]
        ax.barh(range(len(r)), r.stability, color=colors, height=0.7)
        ax.axvline(STABILITY_MIN, color=ps.MUTED, lw=1, ls=(0, (3, 3)))
        ax.set_yticks(range(len(r)), r.label, fontsize=8)
        ax.set_xlim(0, 1.05)
        ax.grid(axis="y", visible=False)
        sign = np.sign(r.lasso_coef.where(r.lasso_coef != 0, r.spearman))
        for i, (v, sg) in enumerate(zip(r.stability, sign)):
            ax.text(v + 0.02, i, "+" if sg > 0 else "−", va="center", fontsize=9, color=ps.INK_2)
        t = r.target.iloc[0]
        ax.set_title(f"{t.capitalize()} change   (CV R²: Lasso {r.lasso_cv_r2.iloc[0]:.2f}, "
                     f"RF {r.rf_cv_r2.iloc[0]:.2f})", fontsize=10)
        ax.set_xlabel("Selection frequency over 300 bootstrap resamples")
    fig.suptitle(title + "\nDark = selected; sign = direction of effect; dashed = 0.6 threshold",
                 x=0.01, ha="left", fontweight="bold", fontsize=11)
    fig.tight_layout()
    ps.save(fig, name)


def main() -> None:
    ps.apply()
    pref, muni = F.load_pref(), F.load_muni()
    out: dict = {}
    tables = []

    # Prefecture, 1985-2015 starts (income starts 1985).
    pr = [run(pref, F.PREF_FEATURES, t, "pref") for t in F.TARGETS]
    figure(pr, "10_feature_stability_pref", "Feature selection, prefectures (1985–2020, within-period)")
    # Prefecture, post-2000 with the university-outflow variable.
    pr2 = [run(pref[pref.period >= 2000], F.PREF_FEATURES_POST2000, t, "pref_post2000") for t in F.TARGETS]
    # Municipality: core features (1995/2005-2020), and with vacancy + women (2005-2015 starts).
    mu = [run(muni, F.MUNI_FEATURES, t, "muni") for t in F.TARGETS]
    figure(mu, "11_feature_stability_muni", "Feature selection, municipalities (within-period)")
    mu2 = [run(muni[muni.period >= 2005], F.MUNI_FEATURES_EXTRA, t, "muni_extra") for t in F.TARGETS]

    for group in (pr, pr2, mu, mu2):
        for r in group:
            tables.append(r)
            out.setdefault(r.level.iloc[0], {})[r.target.iloc[0]] = r[r.selected].feature.tolist()

    allr = pd.concat(tables)
    for lv, g in allr.groupby("level"):
        g.round(3).to_csv(ROOT / "reports" / f"feature_selection_{lv}.csv", index=False)
    (ROOT / "config" / "selected_features.json").write_text(json.dumps(out, indent=1))

    lines = ["# Step 7: feature selection results", "",
             "Generated by `python src/select_features.py`. Method: see the module docstring. "
             "All features and targets are period-demeaned (compared within the same period).", ""]
    for lv, g in allr.groupby("level", sort=False):
        lines.append(f"## {lv}")
        for t, h in g.groupby("target"):
            lines.append(f"\n**{t}**: n = {h.n_obs.iloc[0]}, Lasso CV R² = {h.lasso_cv_r2.iloc[0]:.2f}, "
                         f"RF CV R² = {h.rf_cv_r2.iloc[0]:.2f}\n")
            lines.append("| Feature | Stability | Lasso coef | RF importance | VIF | Spearman | Selected |")
            lines.append("|---|---|---|---|---|---|---|")
            for r in h.itertuples():
                lines.append(f"| {r.label} | {r.stability:.2f} | {r.lasso_coef:+.3f} | {r.rf_importance:.3f} | "
                             f"{r.vif:.1f} | {r.spearman:+.2f} | {'✓' if r.selected else ''} |")
        lines.append("")
    (ROOT / "reports" / "feature_selection.md").write_text("\n".join(lines))
    print(json.dumps(out, indent=1))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile src/models.py
"""Step 8a: explanatory models ("what contributes most") and ANOVA.

Usage:
    python src/models.py

Regressions use the features chosen in step 7 (config/selected_features.json), z-scored so
coefficients are comparable: "+1 standard deviation of X -> +b percentage points of 5-year
change". Standard errors are clustered by area (repeated observations of one area are not
independent).

  Prefecture, spec A: period fixed effects. Compares prefectures within the same period.
  Prefecture, spec B: period + prefecture fixed effects. Uses only changes *within* a
                      prefecture over time (does a prefecture lose more people when its own
                      job market weakens?). Slow-moving variables lose power here.
  Municipality:       period + prefecture fixed effects (compares municipalities within the
                      same prefecture and period). Distance to metro is dropped here: it is a
                      prefecture-level value, so the prefecture fixed effects absorb it.
ANOVA: how much variance in change is explained by region, period, and (municipal) size class.

Outputs: reports/models_explanatory.md, reports/coefficients.csv, figures 12-14.
"""

from __future__ import annotations

import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm
from statsmodels.stats.multicomp import pairwise_tukeyhsd

import features as F
import plotstyle as ps
from estat import ROOT

SEL = json.loads((ROOT / "config" / "selected_features.json").read_text())


def fit(df: pd.DataFrame, target: str, feats: list[str], fe: list[str], cluster: str, extra: list[str] = ()):
    d = df.dropna(subset=feats + [target]).copy()
    for f in feats:
        d[f] = (d[f] - d[f].mean()) / d[f].std()
    rhs = " + ".join(list(feats) + list(extra) + [f"C({x})" for x in fe])
    m = smf.ols(f"{target} ~ {rhs}", data=d).fit(cov_type="cluster", cov_kwds={"groups": d[cluster]})
    ci = m.conf_int()
    out = pd.DataFrame({"coef": m.params, "lo": ci[0], "hi": ci[1], "p": m.pvalues}).loc[list(feats) + list(extra)]
    out["feature"] = out.index
    out["label"] = [F.LABELS.get(f, f) for f in out.index]
    # Within R2: share of variance left after the fixed effects that the features explain.
    base = smf.ols(f"{target} ~ " + " + ".join(f"C({x})" for x in fe), data=d).fit()
    within = 1 - m.ssr / base.ssr
    return out.reset_index(drop=True), {"n": int(m.nobs), "r2": m.rsquared, "within_r2": within}


def coef_plot(blocks: list[tuple[str, pd.DataFrame]], title: str, name: str) -> None:
    """blocks: (series name, coefficient table); up to 2 series per panel, one panel per target."""
    targets = sorted({t for t, _ in [(b[1].target.iloc[0], 0) for b in blocks]}, key=F.TARGETS.index)
    fig, axes = plt.subplots(1, len(targets), figsize=(5.6 * len(targets), 4.8), sharey=False)
    for ax, t in zip(np.atleast_1d(axes), targets):
        bs = [(n, b) for n, b in blocks if b.target.iloc[0] == t]
        feats = list(dict.fromkeys(f for _, b in bs for f in b.feature))
        feats = sorted(feats, key=lambda f: bs[0][1].set_index("feature").coef.get(f, 0))
        y = {f: i for i, f in enumerate(feats)}
        off = np.linspace(-0.16, 0.16, len(bs)) if len(bs) > 1 else [0]
        for k, ((nm, b), o) in enumerate(zip(bs, off)):
            yy = [y[f] + o for f in b.feature]
            ax.hlines(yy, b.lo, b.hi, color=ps.SERIES[k], lw=2)
            ax.scatter(b.coef, yy, color=ps.SERIES[k], s=40, zorder=3, edgecolor=ps.SURFACE, linewidth=2,
                       label=nm)
        ax.axvline(0, color=ps.BASELINE, lw=1)
        ax.set_yticks(range(len(feats)), [F.LABELS.get(f, f) for f in feats], fontsize=8)
        ax.grid(axis="y", visible=False)
        ax.set_title(f"{t.capitalize()} change", fontsize=10)
        ax.set_xlabel("Effect of +1 SD on 5-year change (pp), 95% CI")
    fig.suptitle(title, x=0.01, ha="left", fontweight="bold", fontsize=11)
    fig.tight_layout()
    handles, labels = np.atleast_1d(axes)[0].get_legend_handles_labels()
    if len(handles) > 1:
        fig.subplots_adjust(top=0.82)
        fig.legend(handles, labels, loc="upper left", bbox_to_anchor=(0.01, 0.93), ncol=len(handles), fontsize=9)
    ps.save(fig, name)


def anova(pref: pd.DataFrame, muni: pd.DataFrame, lines: list[str]) -> None:
    rows = []
    for t in ["growth", *F.TARGETS]:
        m = smf.ols(f"{t} ~ C(region) + C(period_label) + C(region):C(period_label)", data=pref).fit()
        a = anova_lm(m, typ=2)
        a["eta_sq"] = a.sum_sq / a.sum_sq.sum()
        for term, r in a.iterrows():
            rows.append({"level": "pref", "target": t, "term": term, "F": r.F, "p": r["PR(>F)"], "eta_sq": r.eta_sq})
    mu = muni.copy()
    mu["size"] = pd.cut(mu.pop_start, [0, 5e3, 2e4, 5e4, 2e5, np.inf],
                        labels=["<5k", "5-20k", "20-50k", "50-200k", "200k+"]).astype(str)
    for t in ["growth", *F.TARGETS]:
        m = smf.ols(f"{t} ~ C(size) + C(region) + C(period_label)", data=mu).fit()
        a = anova_lm(m, typ=2)
        a["eta_sq"] = a.sum_sq / a.sum_sq.sum()
        for term, r in a.iterrows():
            rows.append({"level": "muni", "target": t, "term": term, "F": r.F, "p": r["PR(>F)"], "eta_sq": r.eta_sq})
    an = pd.DataFrame(rows)
    an.to_csv(ROOT / "reports" / "anova.csv", index=False)

    nice = {"C(region)": "Region", "C(period_label)": "Period", "C(region):C(period_label)": "Region × period",
            "C(size)": "Size class", "Residual": "Unexplained"}
    lines += ["## ANOVA: how much of the variation is due to region, period, size?", "",
              "Share of total variance (η², Type II sums of squares). All listed terms p < 0.001 unless noted.", ""]
    for lv, g in an.groupby("level", sort=False):
        piv = g.pivot(index="term", columns="target", values="eta_sq")[["growth", "natural", "social"]]
        pv = g.pivot(index="term", columns="target", values="p")
        lines += [f"**{'Prefectures' if lv == 'pref' else 'Municipalities'}**", "",
                  "| Term | Total growth | Natural | Social |", "|---|---|---|---|"]
        for term in piv.index:
            cells = []
            for t in ["growth", "natural", "social"]:
                p = pv.loc[term, t]
                star = "" if term == "Residual" or p < 0.001 else f" (p={p:.2f})"
                cells.append(f"{piv.loc[term, t]:.2f}{star}")
            lines.append(f"| {nice.get(term, term)} | " + " | ".join(cells) + " |")
        lines.append("")

    # Figure 14: stacked eta^2 per target, prefectures and municipalities.
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), sharey=True)
    for ax, (lv, g) in zip(axes, an.groupby("level", sort=False)):
        terms = [t for t in nice if t in set(g.term)]
        piv = g.pivot(index="target", columns="term", values="eta_sq").loc[["growth", "natural", "social"], terms]
        colors = {"C(region)": ps.SERIES[0], "C(period_label)": ps.SERIES[1], "C(size)": ps.SERIES[2],
                  "C(region):C(period_label)": ps.SERIES[6], "Residual": ps.GRID}
        left = np.zeros(len(piv))
        for term in terms:
            ax.barh(range(len(piv)), piv[term], left=left, color=colors[term], height=0.6,
                    edgecolor=ps.SURFACE, linewidth=2, label=nice[term])
            for i, v in enumerate(piv[term]):
                if v > 0.07:
                    ax.text(left[i] + v / 2, i, f"{v:.0%}", ha="center", va="center", fontsize=8,
                            color=ps.INK if term == "Residual" else "#ffffff")
            left += piv[term].values
        ax.set_yticks(range(len(piv)), ["Total growth", "Natural", "Social"])
        ax.set_xlim(0, 1)
        ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
        ax.grid(axis="y", visible=False)
        ax.set_title("Prefectures" if lv == "pref" else "Municipalities", fontsize=10)
        ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=3, fontsize=8)
        ax.invert_yaxis()
    fig.suptitle("ANOVA: share of variation explained by region, period and size", x=0.01, ha="left",
                 fontweight="bold", fontsize=11)
    fig.tight_layout()
    ps.save(fig, "14_anova_variance_shares")

    # Tukey HSD: which regions differ in social change, municipalities 2015-2020.
    d = mu[mu.period == 2015]
    tk = pairwise_tukeyhsd(d.social, d.region)
    tkd = pd.DataFrame(tk.summary().data[1:], columns=tk.summary().data[0])
    tkd.to_csv(ROOT / "reports" / "tukey_region_social_2015.csv", index=False)
    means = d.groupby("region").social.agg(["mean", "median", "count"]).sort_values("mean")
    lines += ["### Tukey HSD: regions, municipal social change 2015–2020", "",
              "| Region | Mean | Median | n |", "|---|---|---|---|"]
    lines += [f"| {r} | {v['mean']:.2f} | {v['median']:.2f} | {int(v['count'])} |" for r, v in means.iterrows()]
    sig = tkd[tkd.reject == True]  # noqa: E712
    lines += ["", f"{len(sig)} of {len(tkd)} region pairs differ significantly (family-wise α = 0.05). "
              "Full table: `reports/tukey_region_social_2015.csv`.", ""]


def main() -> None:
    ps.apply()
    pref, muni = F.load_pref(), F.load_muni()
    pref = pref[pref.period >= 1985]  # income starts 1985
    lines = ["# Step 8a: explanatory models", "", "Generated by `python src/models.py`. "
             "Coefficients: effect of +1 standard deviation of a feature on the 5-year change, in percentage "
             "points, with standard errors clustered by area. ✱ = p < 0.05.", ""]
    coefs, plots_pref, plots_muni = [], [], []

    for t in F.TARGETS:
        feats = SEL["pref"][t]
        a, sa = fit(pref, t, feats, ["period"], "area_code", extra=["disaster_2011"])
        b, sb = fit(pref, t, feats, ["period", "pref"], "area_code", extra=["disaster_2011"])
        # Distance is measured at prefecture level, so prefecture fixed effects absorb it completely.
        mf = [f for f in SEL["muni"][t] if f != "log_dist_metro"]
        c, sc = fit(muni, t, mf, ["period", "pref"], "area_code")
        for tab, nm, s, lv in [(a, "A: period FE", sa, "pref"), (b, "B: period + prefecture FE", sb, "pref"),
                               (c, "period + prefecture FE", sc, "muni")]:
            tab["target"], tab["spec"], tab["level"] = t, nm, lv
            coefs.append(tab)
            lines += [f"## {'Prefectures' if lv == 'pref' else 'Municipalities'}: {t} change, {nm}", "",
                      f"n = {s['n']}, R² = {s['r2']:.2f}, within-R² (beyond fixed effects) = {s['within_r2']:.2f}", "",
                      "| Feature | Coef (pp per SD) | 95% CI | p |", "|---|---|---|---|"]
            for r in tab.sort_values("coef", key=abs, ascending=False).itertuples():
                lines.append(f"| {r.label} | {r.coef:+.2f}{' ✱' if r.p < 0.05 else ''} | "
                             f"[{r.lo:+.2f}, {r.hi:+.2f}] | {r.p:.3f} |")
            lines.append("")
        plots_pref += [("A: compare prefectures (period FE)", a), ("B: within prefecture (two-way FE)", b)]
        plots_muni += [("municipalities", c)]

    coef_plot(plots_pref, "What drives 5-year population change? Prefectures, 1985–2020", "12_coefficients_pref")
    coef_plot(plots_muni, "What drives 5-year population change? Municipalities, 1995–2020 "
                          "(within prefecture & period)", "13_coefficients_muni")
    pd.concat(coefs).to_csv(ROOT / "reports" / "coefficients.csv", index=False)
    anova(pref, muni, lines)
    (ROOT / "reports" / "models_explanatory.md").write_text("\n".join(lines))
    print("\n".join(lines))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile src/forecast.py
"""Step 8b: forecast population 5 and 10 years ahead, with honest out-of-time backtests.

Usage:
    python src/forecast.py

A forecast made at year t may only use information known at t:
  - features measured at t (start of period), plus the change over the previous period;
  - training rows whose outcome had already happened by t (period + horizon <= t).

Models:
  naive        the last period's change repeats
  ridge        linear model on standardized features (can extrapolate trends)
  gbm          histogram gradient boosting (non-linear; cannot extrapolate beyond training range)
  *_parts      predict natural and social change separately and add them up

Backtests: 5-year horizon tested on periods starting 2005, 2010, 2015; 10-year horizon on
2010 (-> 2020). Then every model is refit on all data to forecast 2020->2025 and 2020->2030.
The 2020->2025 forecast is checked against the official 2024 population estimates.

Outputs: reports/forecast.md, reports/forecast_pref.csv, reports/forecast_muni.csv,
reports/backtest_metrics.csv, figures 15-17.
"""

from __future__ import annotations

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import RidgeCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

import features as F
import plotstyle as ps
from estat import ROOT

PROC = ROOT / "data" / "processed"
PREF_X = ["aging_rate", "young_women_share", "tfr", "unmarried_f_30_34_rate", "income_rel",
          "job_openings_ratio", "unemployment_rate", "did_share", "vacancy_rate", "hs_grads_jobs_other_pref",
          "fiscal_strength", "snow_days_5y", "log_population", "log_dist_metro"]
MUNI_X = ["aging_rate", "income_rel", "unemployment_rate", "did_share", "log_population", "doctors_per_1k",
          "fiscal_strength", "log_dist_metro"]
LAGS = ["lag_growth", "lag_natural", "lag_social"]


def with_lags(panel: pd.DataFrame, future: pd.DataFrame) -> pd.DataFrame:
    d = pd.concat([panel, future], ignore_index=True).sort_values(["area_code", "period"])
    g = d.groupby("area_code")
    for c in ["growth", "natural", "social"]:
        prev = g[c].shift(1)
        same_gap = (d.period - g.period.shift(1)) == 5
        d[f"lag_{c}"] = prev.where(same_gap)
    # 10-year outcome: population at t+10 relative to t.
    nxt = g.pop_end.shift(-1).where((g.period.shift(-1) - d.period) == 5)
    d["growth10"] = (nxt / d.pop_start - 1) * 100
    return d.reset_index(drop=True)


def model(kind: str):
    if kind == "ridge":
        return make_pipeline(SimpleImputer(strategy="median", add_indicator=True), StandardScaler(),
                             RidgeCV(alphas=np.logspace(-2, 3, 30)))
    return HistGradientBoostingRegressor(max_iter=400, learning_rate=0.05, max_leaf_nodes=15,
                                         min_samples_leaf=10, l2_regularization=1.0, random_state=0)


def predict(train: pd.DataFrame, test: pd.DataFrame, X: list[str], horizon: int) -> pd.DataFrame:
    """All model variants for one train/test split. Returns test rows with prediction columns."""
    out = test[["area_code", "period", "pop_start"]].copy()
    tgt = "growth" if horizon == 5 else "growth10"
    tr = train.dropna(subset=[tgt, "lag_growth"])
    if horizon == 5:
        out["naive"] = test.lag_growth
    else:  # repeat last 5-year rate twice (compounded)
        out["naive"] = ((1 + test.lag_growth / 100) ** 2 - 1) * 100
    for kind in ["ridge", "gbm"]:
        out[kind] = model(kind).fit(tr[X], tr[tgt]).predict(test[X])
        if horizon == 5:
            parts = sum(model(kind).fit(tr[X], tr[c]).predict(test[X]) for c in ["natural", "social"])
            out[f"{kind}_parts"] = parts
    return out


def metrics(pred: pd.DataFrame, actual: pd.Series, pop_start: pd.Series, models: list[str]) -> list[dict]:
    rows = []
    for m in models:
        err = pred[m] - actual
        pop_err = pop_start * err / 100
        rows.append({"model": m, "mae_pp": err.abs().mean(), "rmse_pp": np.sqrt((err ** 2).mean()),
                     "bias_pp": err.mean(), "total_pop_error_pct": pop_err.sum() / pop_start.sum() * 100})
    return rows


def backtest(d: pd.DataFrame, X: list[str], level: str, tests5: list[int], tests10: list[int]):
    rows, preds = [], []
    for horizon, tests in [(5, tests5), (10, tests10)]:
        tgt = "growth" if horizon == 5 else "growth10"
        for t in tests:
            train = d[d.period + horizon <= t]
            test = d[(d.period == t)].dropna(subset=[tgt, "lag_growth"])
            p = predict(train, test, X, horizon)
            models = [c for c in p.columns if c not in ("area_code", "period", "pop_start")]
            for r in metrics(p, test[tgt].values, test.pop_start.values, models):
                rows.append({"level": level, "horizon": horizon, "test_period": f"{t}-{t + horizon}",
                             "n_test": len(test), **r})
            p["actual"] = test[tgt].values
            p["horizon"] = horizon
            preds.append(p)
    return pd.DataFrame(rows), pd.concat(preds)


def residual_band(preds: pd.DataFrame, model_name: str, horizon: int) -> tuple[float, float]:
    """80% prediction interval: +/- the 80th percentile of absolute backtest errors.

    Symmetric on purpose. There are only 1-3 backtests per horizon and their bias flips sign
    between periods (the 10-year model over-predicted decline in 2010-2020, while the
    2020-2024 live check shows under-prediction), so an asymmetric band would overfit one period.
    """
    p = preds[preds.horizon == horizon]
    q = float((p["actual"] - p[model_name]).abs().quantile(0.8))
    return -q, q


def main() -> None:
    ps.apply()
    pref = with_lags(F.load_pref(), F.add_transforms(pd.read_parquet(PROC / "pref_start_2020.parquet")))
    pref = pref[pref.period >= 1985]
    muni = with_lags(F.load_muni(), F.add_transforms(pd.read_parquet(PROC / "muni_start_2020.parquet")))

    mp, pp = backtest(pref, PREF_X + LAGS, "pref", [2005, 2010, 2015], [2010])
    mm, pm = backtest(muni, MUNI_X + LAGS, "muni", [2010, 2015], [2010])
    met = pd.concat([mp, mm])
    met.to_csv(ROOT / "reports" / "backtest_metrics.csv", index=False)

    # Pick the best model per level and horizon by mean MAE across backtests.
    best = (met.groupby(["level", "horizon", "model"]).mae_pp.mean().reset_index()
            .sort_values("mae_pp").groupby(["level", "horizon"]).head(1).set_index(["level", "horizon"]).model)
    print(met.round(2).to_string(index=False))
    print("best:", best.to_dict())

    # Final forecasts from 2020.
    out = {}
    for level, d, X, preds in [("pref", pref, PREF_X + LAGS, pp), ("muni", muni, MUNI_X + LAGS, pm)]:
        future = d[d.period == 2020]
        p5 = predict(d[d.period + 5 <= 2020], future, X, 5)
        p10 = predict(d[d.period + 10 <= 2020], future, X, 10)
        m5, m10 = best[(level, 5)], best[(level, 10)]
        lo5, hi5 = residual_band(preds, m5, 5)
        lo10, hi10 = residual_band(preds, m10, 10)
        f = future[["area_code", "pop_start"]].copy()
        if "pref_name" in future:
            f["name"] = future.pref_name.values
        if "area_name" in future and level == "muni":
            f["name"] = future.area_name.values
        f["growth_2025"], f["growth_2025_lo"], f["growth_2025_hi"] = p5[m5].values, p5[m5].values + lo5, p5[m5].values + hi5
        f["growth_2030"], f["growth_2030_lo"], f["growth_2030_hi"] = p10[m10].values, p10[m10].values + lo10, p10[m10].values + hi10
        f["pop_2025"] = f.pop_start * (1 + f.growth_2025 / 100)
        f["pop_2030"] = f.pop_start * (1 + f.growth_2030 / 100)
        f["model_5y"], f["model_10y"] = m5, m10
        out[level] = f

    # Live check: 2020 -> 2024 official estimates (4 of the 5 forecast years).
    y = pd.read_parquet(PROC / "pref_yearly.parquet")
    p24 = y[y.year == 2024].set_index("area_code").population
    fp = out["pref"]
    fp["actual_2024"] = fp.area_code.map(p24)
    fp["actual_growth_2020_2024"] = (fp.actual_2024 / fp.pop_start - 1) * 100
    fp["pred_growth_2020_2024"] = ((1 + fp.growth_2025 / 100) ** 0.8 - 1) * 100  # 4/5 of the period
    live_mae = (fp.pred_growth_2020_2024 - fp.actual_growth_2020_2024).abs().mean()
    live_corr = fp.pred_growth_2020_2024.corr(fp.actual_growth_2020_2024)
    fp.round(3).to_csv(ROOT / "reports" / "forecast_pref.csv", index=False)
    out["muni"].round(3).to_csv(ROOT / "reports" / "forecast_muni.csv", index=False)

    figures(met, pp, fp, best)

    # Report.
    L = ["# Step 8b: forecasting", "", "Generated by `python src/forecast.py`; see the module docstring for the method.", "",
         "## Backtests (out-of-time)", "",
         "MAE = mean absolute error of the predicted 5- or 10-year population change, in percentage points. "
         "Bias = mean error (negative = predicted too much decline). "
         "Total error = error in the summed population of all test areas.", ""]
    for (lv, h), g in met.groupby(["level", "horizon"], sort=False):
        L += [f"### {'Prefectures' if lv == 'pref' else 'Municipalities'}, {h}-year horizon", "",
              "| Model | " + " | ".join(f"MAE {t}" for t in g.test_period.unique()) + " | Mean MAE | Mean bias |",
              "|---|" + "---|" * (g.test_period.nunique() + 2)]
        for m, gm in g.groupby("model"):
            cells = " | ".join(f"{v:.2f}" for v in gm.mae_pp)
            mark = " **(best)**" if best[(lv, h)] == m else ""
            L.append(f"| {m}{mark} | {cells} | {gm.mae_pp.mean():.2f} | {gm.bias_pp.mean():+.2f} |")
        L.append("")
    nat = fp.pop_start.sum()
    L += ["## Forecast from 2020 (prefectures, excluding Fukushima)", "",
          f"Total for the 46 prefectures: {nat / 1e6:.2f}M (2020) → {fp.pop_2025.sum() / 1e6:.2f}M (2025) → "
          f"{fp.pop_2030.sum() / 1e6:.2f}M (2030).", "",
          f"**Live check against 2024 official estimates:** MAE {live_mae:.2f} pp over 2020–2024, "
          f"correlation {live_corr:.2f} between predicted and actual change.", "",
          "| Prefecture | 2020 | 2025 forecast (80% interval) | 2030 forecast (80% interval) | Actual 2020→24 | Predicted 2020→24 |",
          "|---|---|---|---|---|---|"]
    for r in fp.sort_values("growth_2030").itertuples():
        L.append(f"| {r.name} | {r.pop_start / 1e3:,.0f}k | {r.growth_2025:+.1f}% [{r.growth_2025_lo:+.1f}, {r.growth_2025_hi:+.1f}] | "
                 f"{r.growth_2030:+.1f}% [{r.growth_2030_lo:+.1f}, {r.growth_2030_hi:+.1f}] | "
                 f"{r.actual_growth_2020_2024:+.1f}% | {r.pred_growth_2020_2024:+.1f}% |")
    fm = out["muni"]
    L += ["", "## Forecast from 2020 (municipalities)", "",
          f"{len(fm)} municipalities. Predicted to shrink 2020→2025: {(fm.growth_2025 < 0).mean():.0%}; "
          f"shrink by more than 10% by 2030: {(fm.growth_2030 < -10).mean():.0%}. Full table: `reports/forecast_muni.csv`.", ""]
    (ROOT / "reports" / "forecast.md").write_text("\n".join(L))
    print("\n".join(L[-60:]))


def figures(met: pd.DataFrame, pp: pd.DataFrame, fp: pd.DataFrame, best) -> None:
    # 15: backtest MAE by model (5-year, both levels).
    g = met[met.horizon == 5].groupby(["level", "model"]).mae_pp.mean().unstack(0)
    order = ["naive", "ridge", "ridge_parts", "gbm", "gbm_parts"]
    g = g.loc[order]
    fig, axes = plt.subplots(1, 2, figsize=(9, 3.2), sharey=True)
    for ax, lv, title in zip(axes, ["pref", "muni"], ["Prefectures", "Municipalities"]):
        v = g[lv]
        colors = [ps.SERIES[0] if m == best[(lv, 5)] else ps.BLUE_RAMP[1] for m in v.index]
        ax.barh(range(len(v)), v, color=colors, height=0.65)
        for i, x in enumerate(v):
            ax.text(x + v.max() * 0.02, i, f"{x:.2f}", va="center", fontsize=8, color=ps.INK_2)
        ax.set_yticks(range(len(v)), v.index)
        ax.set_xlim(0, v.max() * 1.2)
        ax.grid(axis="y", visible=False)
        ax.set_title(title, fontsize=10)
        ax.set_xlabel("Mean absolute error (pp)")
        ax.invert_yaxis()
    fig.suptitle("5-year forecast error in out-of-time backtests (dark = best)", x=0.01, ha="left",
                 fontweight="bold", fontsize=11)
    fig.tight_layout()
    ps.save(fig, "15_backtest_errors")

    # 16: predicted vs actual, prefectures, test 2015-2020, best 5-year model.
    m = best[("pref", 5)]
    p = pp[(pp.horizon == 5) & (pp.period == 2015)]
    fig, ax = plt.subplots(figsize=(5.2, 5))
    lim = [min(p.actual.min(), p[m].min()) - 0.5, max(p.actual.max(), p[m].max()) + 0.5]
    ax.plot(lim, lim, color=ps.BASELINE, lw=1)
    ax.scatter(p[m], p.actual, s=28, color=ps.SERIES[0], edgecolor=ps.SURFACE, linewidth=1.5, zorder=3)
    names = F.load_pref().drop_duplicates("area_code").set_index("area_code").pref_name
    for r in p.itertuples():
        if abs(getattr(r, m) - r.actual) > 0.8 or r.actual > 2 or r.actual < -5.5:
            ax.annotate(names[r.area_code], (getattr(r, m), r.actual), xytext=(4, 2), textcoords="offset points",
                        fontsize=7, color=ps.INK_2)
    ax.set_xlim(lim)
    ax.set_ylim(lim)
    ax.set_xlabel(f"Predicted change 2015–2020 (%), model: {m}")
    ax.set_ylabel("Actual change 2015–2020 (%)")
    ax.set_title("Backtest: forecast made in 2015 vs. what happened")
    ps.save(fig, "16_backtest_pref_2015")

    # 17: 2030 forecast per prefecture with 80% interval, plus 2020-24 actual (scaled) as check.
    f = fp.sort_values("growth_2030")
    fig, ax = plt.subplots(figsize=(7.5, 10))
    y = np.arange(len(f))
    ax.hlines(y, f.growth_2030_lo, f.growth_2030_hi, color=ps.BLUE_RAMP[1], lw=4)
    ax.scatter(f.growth_2030, y, color=ps.SERIES[0], s=30, zorder=3, edgecolor=ps.SURFACE, linewidth=1.5,
               label="Forecast 2020→2030 (80% interval)")
    ax.scatter(f.growth_2025, y, color=ps.SERIES[1], s=22, zorder=3, marker="D", edgecolor=ps.SURFACE,
               linewidth=1, label="Forecast 2020→2025")
    ax.axvline(0, color=ps.BASELINE, lw=1)
    ax.set_yticks(y, f.name, fontsize=8)
    ax.grid(axis="y", visible=False)
    ax.set_xlabel("Population change from 2020 (%)")
    ax.set_title("Forecast population change by prefecture", pad=28)
    ax.legend(loc="upper left", bbox_to_anchor=(0, 1.035), ncol=2, fontsize=8)
    ps.save(fig, "17_forecast_pref")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile src/time_varying.py
"""Step 9: do the most important factors change over time?

Usage:
    python src/time_varying.py

For each 5-year period separately, regress natural and social change on one fixed set of
features (z-scored within the period, so coefficients compare across periods):
  - Shapley R² decomposition (dominance analysis): each feature's share of the explained
    variance, averaged over all orders of entry. Robust to correlated features, which matters
    with 47 prefectures.
  - Standardized coefficients with 95% bootstrap intervals (resampling areas).
  - An F-test of "coefficients are the same in every period": pooled model with
    feature x period interactions vs. without (both with period fixed effects).

Outputs: reports/time_varying.md, reports/time_varying_<level>.csv, figures 18-20.
"""

from __future__ import annotations

from itertools import combinations
from math import factorial

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm

import features as F
import plotstyle as ps
from estat import ROOT

RNG = np.random.default_rng(7)
N_BOOT = 500
SETS = {
    ("pref", "social"): ["income_rel", "job_openings_ratio", "unemployment_rate", "aging_rate",
                         "log_dist_metro", "snow_days_5y"],
    ("pref", "natural"): ["aging_rate", "tfr", "unmarried_f_30_34_rate", "income_rel", "vacancy_rate"],
    ("muni", "social"): ["aging_rate", "fiscal_strength", "income_rel", "log_population", "did_share",
                         "unemployment_rate"],
    ("muni", "natural"): ["aging_rate", "did_share", "income_rel", "log_population", "unemployment_rate"],
}
# README framing: roughly 30, 20, 10 years before the latest period, and now.
ERA = {1985: "~30 yrs ago", 1995: "~20 yrs ago", 2005: "~10 yrs ago", 2015: "now"}


def r2(X: np.ndarray, y: np.ndarray) -> float:
    if X.shape[1] == 0:
        return 0.0
    A = np.column_stack([np.ones(len(y)), X])
    beta, *_ = np.linalg.lstsq(A, y, rcond=None)
    resid = y - A @ beta
    return 1 - resid @ resid / ((y - y.mean()) @ (y - y.mean()))


def shapley_r2(X: pd.DataFrame, y: pd.Series) -> pd.Series:
    """Exact Shapley decomposition of R² (fine for <= 8 features)."""
    cols = list(X.columns)
    k = len(cols)
    Xv, yv = X.values, y.values
    cache: dict[frozenset, float] = {}

    def val(s: frozenset) -> float:
        if s not in cache:
            cache[s] = r2(Xv[:, sorted(cols.index(c) for c in s)], yv)
        return cache[s]

    out = {}
    for c in cols:
        others = [o for o in cols if o != c]
        tot = 0.0
        for m in range(k):
            w = factorial(m) * factorial(k - m - 1) / factorial(k)
            for S in combinations(others, m):
                s = frozenset(S)
                tot += w * (val(s | {c}) - val(s))
        out[c] = tot
    return pd.Series(out)


def per_period(df: pd.DataFrame, level: str, target: str) -> pd.DataFrame:
    feats = SETS[(level, target)]
    rows = []
    for per, g in df.dropna(subset=feats + [target]).groupby("period"):
        X = (g[feats] - g[feats].mean()) / g[feats].std()
        y = g[target]
        A = np.column_stack([np.ones(len(y)), X.values])
        beta = np.linalg.lstsq(A, y.values, rcond=None)[0][1:]
        boots = []
        n = len(g)
        for _ in range(N_BOOT):
            i = RNG.integers(0, n, n)
            Xb = X.values[i]
            Xb = (Xb - Xb.mean(0)) / Xb.std(0, ddof=1)
            boots.append(np.linalg.lstsq(np.column_stack([np.ones(n), Xb]), y.values[i], rcond=None)[0][1:])
        lo, hi = np.percentile(boots, [2.5, 97.5], axis=0)
        sh = shapley_r2(X, y)
        total = r2(X.values, y.values)
        for j, f in enumerate(feats):
            rows.append({"level": level, "target": target, "period": per, "feature": f,
                         "label": F.LABELS.get(f, f), "coef": beta[j], "lo": lo[j], "hi": hi[j],
                         "shapley_r2": sh[f], "share_of_r2": sh[f] / total, "r2": total, "n": n})
    return pd.DataFrame(rows)


def stability_test(df: pd.DataFrame, level: str, target: str) -> dict:
    feats = SETS[(level, target)]
    d = df.dropna(subset=feats + [target]).copy()
    for f in feats:  # standardize within period, as in the per-period fits
        d[f] = d.groupby("period")[f].transform(lambda s: (s - s.mean()) / s.std())
    base = smf.ols(f"{target} ~ C(period) + " + " + ".join(feats), data=d).fit()
    full = smf.ols(f"{target} ~ C(period) * (" + " + ".join(feats) + ")", data=d).fit()
    a = anova_lm(base, full)
    return {"F": float(a.F.iloc[1]), "p": float(a["Pr(>F)"].iloc[1]), "df": int(a.df_diff.iloc[1])}


def plot_shares(res: pd.DataFrame, name: str, title: str) -> None:
    targets = [t for t in F.TARGETS if t in set(res.target)]
    # Colour follows the feature: features in both panels get the same slot in each, and are
    # stacked first; panel-specific features take the remaining slots.
    per_target = {t: list(dict.fromkeys(res[res.target == t].feature)) for t in targets}
    shared = [f for f in per_target[targets[0]] if all(f in per_target[t] for t in targets)]
    fig, axes = plt.subplots(1, len(targets), figsize=(5.4 * len(targets), 4.6))
    for ax, t in zip(np.atleast_1d(axes), targets):
        g = res[res.target == t]
        feats = shared + [f for f in per_target[t] if f not in shared]
        piv = g.pivot(index="period", columns="feature", values="shapley_r2")[feats]
        x = np.arange(len(piv))
        bottom = np.zeros(len(piv))
        for k, f in enumerate(feats):
            v = piv[f].clip(lower=0).values
            ax.bar(x, v, bottom=bottom, color=ps.SERIES[k], width=0.7, edgecolor=ps.SURFACE, linewidth=2,
                   label=F.LABELS.get(f, f))
            for i in range(len(v)):
                if v[i] > 0.09:
                    ax.text(x[i], bottom[i] + v[i] / 2, f"{v[i]:.2f}", ha="center", va="center", fontsize=7,
                            color=ps.INK if ps.SERIES[k] == "#eda100" else "#ffffff")
            bottom += v
        ax.set_xticks(x, [f"{p}–{p + 5}" for p in piv.index], rotation=45, ha="right", fontsize=8)
        ax.set_ylim(0, 1)
        ax.set_ylabel("R² explained (Shapley share per feature)")
        ax.set_title(f"{t.capitalize()} change", fontsize=10)
        ax.grid(axis="x", visible=False)
        ax.legend(loc="upper left", bbox_to_anchor=(1.0, 1.0), fontsize=8)
    fig.suptitle(title, x=0.01, ha="left", fontweight="bold", fontsize=11)
    fig.tight_layout()
    ps.save(fig, name)


def plot_coefs(res: pd.DataFrame, name: str, title: str) -> None:
    feats = list(dict.fromkeys(res.feature))
    ncol = 3
    nrow = int(np.ceil(len(feats) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(10, 2.6 * nrow), sharey=True)
    for ax, f in zip(axes.flat, feats):
        g = res[res.feature == f].sort_values("period")
        x = g.period + 2.5
        ax.fill_between(x, g.lo, g.hi, color=ps.BLUE_RAMP[0], lw=0)
        ax.plot(x, g.coef, color=ps.SERIES[0], marker="o", markersize=6, markeredgecolor=ps.SURFACE,
                markeredgewidth=1.5)
        ax.axhline(0, color=ps.BASELINE, lw=1)
        ax.set_title(F.LABELS.get(f, f), fontsize=9, fontweight="normal")
        ax.tick_params(labelsize=8)
    for ax in list(axes.flat)[len(feats):]:
        ax.set_visible(False)
    fig.supylabel("Effect of +1 SD (pp of 5-yr change)", fontsize=9, color=ps.INK_2)
    fig.suptitle(title + " (95% bootstrap band; x = period midpoint)", x=0.01, ha="left",
                 fontweight="bold", fontsize=11)
    fig.tight_layout()
    ps.save(fig, name)


def main() -> None:
    ps.apply()
    pref = F.load_pref()
    pref = pref[pref.period >= 1985]
    muni = F.load_muni()
    allres, tests = [], []
    for (level, target), _ in SETS.items():
        df = pref if level == "pref" else muni
        r = per_period(df, level, target)
        allres.append(r)
        tests.append({"level": level, "target": target, **stability_test(df, level, target)})
    res = pd.concat(allres)
    for lv, g in res.groupby("level"):
        g.round(4).to_csv(ROOT / "reports" / f"time_varying_{lv}.csv", index=False)
    tests = pd.DataFrame(tests)

    plot_shares(res[res.level == "pref"], "18_shapley_by_period_pref",
                "Which factors explain prefecture differences, period by period?")
    plot_coefs(res[(res.level == "pref") & (res.target == "social")], "19_coef_paths_pref_social",
               "Effect on prefecture social change (net migration) over time")
    plot_shares(res[res.level == "muni"], "20_shapley_by_period_muni",
                "Which factors explain municipal differences, period by period?")

    L = ["# Step 9: do the main factors change over time?", "",
         "Generated by `python src/time_varying.py`; method in the module docstring.", "",
         "## Are the coefficients the same in every period? (F-test, interactions vs. none)", "",
         "| Level | Target | F | df | p | Verdict |", "|---|---|---|---|---|---|"]
    for t in tests.itertuples():
        verdict = "factors **change** over time" if t.p < 0.05 else "no significant change"
        L.append(f"| {t.level} | {t.target} | {t.F:.2f} | {t.df} | {t.p:.4f} | {verdict} |")
    L.append("")
    for (lv, tg), g in res.groupby(["level", "target"], sort=False):
        L += [f"## {'Prefectures' if lv == 'pref' else 'Municipalities'}: {tg} change", "",
              "Shapley share of R² by period (bold = largest share that period); last row = total R².", ""]
        piv = g.pivot(index="label", columns="period", values="share_of_r2")
        coef = g.pivot(index="label", columns="period", values="coef")
        L.append("| Feature | " + " | ".join(f"{p}–{p + 5}" + (f" ({ERA[p]})" if p in ERA else "") for p in piv.columns) + " |")
        L.append("|---|" + "---|" * len(piv.columns))
        top = piv.idxmax()
        for lab in piv.index:
            cells = []
            for p in piv.columns:
                s = f"{piv.loc[lab, p]:.0%} ({coef.loc[lab, p]:+.2f})"
                cells.append(f"**{s}**" if top[p] == lab else s)
            L.append(f"| {lab} | " + " | ".join(cells) + " |")
        r2s = g.groupby("period").r2.first()
        L.append("| *Total R²* | " + " | ".join(f"{v:.2f}" for v in r2s) + " |")
        L += ["", "Cells: share of explained variance (standardized coefficient in pp per SD).", ""]
    (ROOT / "reports" / "time_varying.md").write_text("\n".join(L))
    print("\n".join(L))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile src/build_report.py
"""Step 10: assemble the shareable report page from the pipeline outputs.

Usage:
    python src/build_report.py      # -> reports/site/index.html

Interactive charts are drawn in the browser from data embedded in the page; denser
multi-panel figures are embedded as the PNGs from reports/figures/.
"""

from __future__ import annotations

import base64
import json

import pandas as pd

import geo
from estat import ROOT

REP = ROOT / "reports"
PROC = ROOT / "data" / "processed"
TEMPLATE = ROOT / "src" / "report_template.html"


def img(name: str) -> str:
    return "data:image/png;base64," + base64.b64encode((REP / "figures" / f"{name}.png").read_bytes()).decode()


def data() -> dict:
    y = pd.read_parquet(PROC / "pref_yearly.parquet")
    p = y[y.area_code != "00000"].copy()
    bloc = {**{c: "Tokyo area" for c in geo.TOKYO_AREA},
            **{c: "Osaka & Nagoya areas" for c in ["21", "23", "24", "26", "27", "28", "29"]}}
    p["bloc"] = p.pref.map(bloc).fillna("Rest of Japan")
    g = p.groupby(["bloc", "year"])[["moves_in", "moves_out", "population"]].sum(min_count=1)
    g["rate"] = (g.moves_in - g.moves_out) / g.population * 1000
    migration = [{"name": b, "points": [[int(yr), round(v, 2)] for yr, v in g.loc[b].rate.dropna().items() if yr <= 2024]}
                 for b in ["Tokyo area", "Osaka & Nagoya areas", "Rest of Japan"]]

    peaks = pd.read_csv(REP / "prefecture_peaks.csv")
    peaks = [{"name": r.pref_name, "region": r.region, "peak": int(r.peak_year),
              "since": round(r.change_since_peak_pct, 1), "growing": bool(r.still_growing)}
             for r in peaks.sort_values(["peak_year", "pref_name"]).itertuples()]

    fc = pd.read_csv(REP / "forecast_pref.csv")
    forecast = [{"name": r.name, "pop2020": int(r.pop_start), "g25": round(r.growth_2025, 2),
                 "g30": round(r.growth_2030, 2), "lo30": round(r.growth_2030_lo, 2), "hi30": round(r.growth_2030_hi, 2),
                 "lo25": round(r.growth_2025_lo, 2), "hi25": round(r.growth_2025_hi, 2),
                 "pop2030": int(r.pop_2030), "act24": round(r.actual_growth_2020_2024, 2),
                 "pred24": round(r.pred_growth_2020_2024, 2)}
                for r in fc.sort_values("growth_2030").itertuples()]

    tv = pd.read_csv(REP / "time_varying_pref.csv")
    shap = {}
    for t, gt in tv.groupby("target"):
        feats = list(dict.fromkeys(gt.feature))
        shap[t] = {"periods": [f"{p}–{p + 5}" for p in sorted(gt.period.unique())],
                   "r2": [round(v, 3) for v in gt.groupby("period").r2.first()],
                   "series": [{"name": gt[gt.feature == f].label.iloc[0],
                               "values": [round(max(v, 0), 3) for v in gt[gt.feature == f].sort_values("period").shapley_r2],
                               "coefs": [round(v, 2) for v in gt[gt.feature == f].sort_values("period").coef]}
                              for f in feats]}

    co = pd.read_csv(REP / "coefficients.csv")
    coefs = {}
    for (lv, t), gc in co.groupby(["level", "target"]):
        coefs[f"{lv}_{t}"] = [{"spec": s, "rows": [{"label": r.label, "coef": round(r.coef, 3), "lo": round(r.lo, 3),
                                                     "hi": round(r.hi, 3), "p": round(r.p, 4)} for r in gs.itertuples()]}
                               for s, gs in gc.groupby("spec", sort=False)]

    eda = json.loads((REP / "eda_summary.json").read_text())
    met = pd.read_csv(REP / "backtest_metrics.csv")
    best5 = met[(met.level == "pref") & (met.horizon == 5)].groupby("model").mae_pp.mean()
    return {"migration": migration, "peaks": peaks, "forecast": forecast, "shapley": shap, "coefs": coefs,
            "facts": {"peak_year": eda["national_peak_year"], "peak_pop": eda["national_peak_population"],
                      "first_nat_dec": eda["national_first_natural_decrease_year"],
                      "prefs_declining_now": eda["prefs_declining_by_period"]["2015-2020"],
                      "muni_shrinking": eda["municipal_shares_pct"]["2015-2020"]["shrinking"],
                      "pop2020": int(fc.pop_start.sum()), "pop2030": int(fc.pop_2030.sum()),
                      "mae_best": round(best5.min(), 2), "mae_naive": round(best5["naive"], 2)}}


def main() -> None:
    html = TEMPLATE.read_text()
    html = html.replace("/*__DATA__*/null", json.dumps(data(), ensure_ascii=False))
    for name in ["05_prefecture_growth_heatmap", "06_natural_vs_social", "07_municipal_shrinking_share",
                 "08_municipal_change_by_size", "14_anova_variance_shares", "16_backtest_pref_2015",
                 "20_shapley_by_period_muni", "13_coefficients_muni"]:
        html = html.replace(f"__IMG_{name}__", img(name))
    out = REP / "site" / "index.html"
    out.parent.mkdir(parents=True, exist_ok=True)
    out.write_text(html)
    print(f"wrote {out.relative_to(ROOT)} ({out.stat().st_size / 1e6:.2f} MB)")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile src/report_template.html
<title>Japan's Shrinking Regions</title>
<meta name="claude:viewport-fit-sides" content="cover">
<link rel="preconnect" href="https://fonts.googleapis.com">
<link rel="preconnect" href="https://fonts.gstatic.com" crossorigin>
<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Zen+Kaku+Gothic+New:wght@500;700&family=Noto+Sans+JP:wght@400;500;700&family=IBM+Plex+Mono:wght@400;500&display=swap">
<style>
/* Layout: one reading column (~68ch) that steps out to a wider measure for charts; each research question is a section. */
:root {
  --bg: #f4f5f7; --surface: #fcfcfb; --ink: #141a26; --ink-2: #4a5263; --muted: #7b8394;
  --rule: #dde0e6; --grid: #e4e6eb; --base: #c3c6cf; --accent: #2b4c8c; --band: #cde2fb;
  --s1: #2a78d6; --s2: #eb6834; --s3: #1baf7a; --s4: #eda100; --s5: #e87ba4; --s6: #008300; --s7: #6250d6; --s8: #e34948;
  --plate: #fcfcfb; --tip-bg: #141a26; --tip-ink: #f4f5f7;
  --f-display: "Zen Kaku Gothic New", "Hiragino Sans", "Noto Sans JP", system-ui, sans-serif;
  --f-body: "Noto Sans JP", "Hiragino Sans", system-ui, -apple-system, "Segoe UI", sans-serif;
  --f-data: "IBM Plex Mono", ui-monospace, "SFMono-Regular", Menlo, monospace;
}
@media (prefers-color-scheme: dark) {
  :root:not([data-theme="light"]) {
    --bg: #0f1218; --surface: #171b22; --ink: #eef0f4; --ink-2: #c2c7d2; --muted: #8a91a0;
    --rule: #2a303b; --grid: #262b34; --base: #3d4350; --accent: #8fb0ee; --band: #1f3a63;
    --s1: #3987e5; --s2: #d95926; --s3: #199e70; --s4: #c98500; --s5: #d55181; --s6: #0ca30c; --s7: #9085e9; --s8: #e66767;
    --tip-bg: #eef0f4; --tip-ink: #141a26; color-scheme: dark;
  }
}
:root[data-theme="dark"] {
  --bg: #0f1218; --surface: #171b22; --ink: #eef0f4; --ink-2: #c2c7d2; --muted: #8a91a0;
  --rule: #2a303b; --grid: #262b34; --base: #3d4350; --accent: #8fb0ee; --band: #1f3a63;
  --s1: #3987e5; --s2: #d95926; --s3: #199e70; --s4: #c98500; --s5: #d55181; --s6: #0ca30c; --s7: #9085e9; --s8: #e66767;
  --tip-bg: #eef0f4; --tip-ink: #141a26; color-scheme: dark;
}
* { box-sizing: border-box; }
body {
  background: var(--bg); color: var(--ink); font-family: var(--f-body); font-size: 16px; line-height: 1.65;
  padding-left: max(16px, env(safe-area-inset-left, 0px)); padding-right: max(16px, env(safe-area-inset-right, 0px));
  padding-block: 0 4rem;
}
.wrap { max-width: 1040px; margin: 0 auto; }
.col { max-width: 68ch; }
h1, h2, h3 { font-family: var(--f-display); text-wrap: balance; line-height: 1.25; margin: 0; }
h1 { font-size: clamp(2rem, 5vw, 3.1rem); font-weight: 700; letter-spacing: -0.01em; }
h2 { font-size: 1.6rem; font-weight: 700; }
h3 { font-size: 1.1rem; font-weight: 700; margin-top: 2rem; }
p { margin: 0; }
.stack { display: flex; flex-direction: column; gap: 1rem; }
.eyebrow { font-family: var(--f-data); font-size: 0.78rem; letter-spacing: 0.08em; text-transform: uppercase; color: var(--muted); }
.jp { font-family: var(--f-body); color: var(--muted); font-size: 0.9em; font-weight: 400; margin-left: 0.4em; }
header { padding-block: 3.5rem 2rem; border-bottom: 1px solid var(--rule); display: flex; flex-direction: column; gap: 1.1rem; }
.lede { font-size: 1.15rem; color: var(--ink-2); max-width: 60ch; }
.facts { display: grid; grid-template-columns: repeat(4, minmax(0, 1fr)); gap: 0; margin-top: 1rem; border-top: 1px solid var(--rule); }
.fact { padding: 1rem 1rem 0 0; display: flex; flex-direction: column; gap: 0.2rem; }
.fact + .fact { padding-left: 1rem; border-left: 1px solid var(--rule); }
.fact b { font-family: var(--f-data); font-weight: 500; font-size: 1.7rem; font-variant-numeric: tabular-nums; color: var(--ink); line-height: 1.2; }
.fact span { font-size: 0.85rem; color: var(--ink-2); line-height: 1.4; }
section { padding-block: 3rem 0; display: flex; flex-direction: column; gap: 1.2rem; }
.q { font-family: var(--f-data); font-size: 0.85rem; color: var(--accent); font-weight: 500; }
.answer { font-size: 1.12rem; color: var(--ink); max-width: 64ch; }
ul.tight { margin: 0; padding-left: 1.2rem; display: flex; flex-direction: column; gap: 0.35rem; }
.chart { background: var(--surface); border: 1px solid var(--rule); border-radius: 10px; padding: 1rem 1rem 0.6rem; display: flex; flex-direction: column; gap: 0.6rem; min-width: 0; }
.chart-head { display: flex; flex-wrap: wrap; gap: 0.6rem 1.2rem; align-items: baseline; justify-content: space-between; }
.chart-title { font-weight: 700; font-size: 0.98rem; }
.chart-note { font-size: 0.8rem; color: var(--muted); }
.legend { display: flex; flex-wrap: wrap; gap: 0.3rem 1rem; font-size: 0.82rem; color: var(--ink-2); }
.legend i { display: inline-block; width: 14px; height: 4px; border-radius: 2px; margin-right: 0.4rem; vertical-align: middle; }
.legend i.dot { width: 9px; height: 9px; border-radius: 50%; }
.legend i.dia { width: 8px; height: 8px; border-radius: 1px; transform: rotate(45deg); }
.legend i.band { height: 8px; }
.svgbox { width: 100%; overflow-x: auto; }
svg { display: block; font-family: var(--f-body); }
.toggle { display: inline-flex; border: 1px solid var(--rule); border-radius: 999px; padding: 2px; gap: 2px; }
.toggle button { font: inherit; font-size: 0.82rem; border: 0; background: transparent; color: var(--ink-2); padding: 0.25rem 0.8rem; border-radius: 999px; cursor: pointer; }
.toggle button[aria-pressed="true"] { background: var(--ink); color: var(--bg); }
.toggle button:focus-visible, summary:focus-visible { outline: 2px solid var(--accent); outline-offset: 2px; }
figure { margin: 0; display: flex; flex-direction: column; gap: 0.5rem; min-width: 0; }
figure img { background: var(--plate); border-radius: 8px; border: 1px solid var(--rule); padding: 0.5rem; width: 100%; height: auto; }
figcaption { font-size: 0.82rem; color: var(--muted); max-width: 70ch; }
.duo { display: grid; grid-template-columns: repeat(2, minmax(0, 1fr)); gap: 1.2rem; align-items: start; }
details { border-top: 1px solid var(--rule); padding-top: 0.6rem; }
summary { cursor: pointer; font-size: 0.88rem; color: var(--accent); }
.tbl { overflow-x: auto; margin-top: 0.6rem; }
table { border-collapse: collapse; font-size: 0.82rem; width: 100%; font-variant-numeric: tabular-nums; }
th, td { padding: 0.3rem 0.6rem; text-align: right; border-bottom: 1px solid var(--rule); white-space: nowrap; }
th:first-child, td:first-child { text-align: left; }
th { font-weight: 500; color: var(--ink-2); }
.era { display: grid; grid-template-columns: repeat(4, minmax(0, 1fr)); gap: 0.8rem; }
.era div { border-top: 3px solid var(--base); padding-top: 0.6rem; display: flex; flex-direction: column; gap: 0.3rem; }
.era div.now { border-top-color: var(--accent); }
.era b { font-family: var(--f-data); font-size: 0.8rem; color: var(--muted); font-weight: 500; }
.era p { font-size: 0.92rem; }
.limits { font-size: 0.95rem; color: var(--ink-2); }
footer { margin-top: 3.5rem; padding-top: 1.2rem; border-top: 1px solid var(--rule); font-size: 0.85rem; color: var(--muted); display: flex; flex-direction: column; gap: 0.5rem; }
code { font-family: var(--f-data); font-size: 0.85em; }
#tip { position: fixed; pointer-events: none; background: var(--tip-bg); color: var(--tip-ink); font-size: 0.8rem; line-height: 1.45; padding: 0.45rem 0.6rem; border-radius: 6px; max-width: 260px; z-index: 20; box-shadow: 0 4px 14px rgba(0,0,0,.18); font-variant-numeric: tabular-nums; }
#tip b { font-weight: 700; }
@media (max-width: 720px) {
  .facts { grid-template-columns: repeat(2, minmax(0, 1fr)); }
  .fact:nth-child(3) { padding-left: 0; border-left: 0; }
  .duo, .era { grid-template-columns: minmax(0, 1fr); }
}
@media (prefers-reduced-motion: no-preference) { .toggle button { transition: background .15s, color .15s; } }
</style>

<div class="wrap">
<header>
  <div class="eyebrow">JAIST K417EJ capstone · e-Stat SSDS 1975–2024 · 47 prefectures, 1,741 municipalities</div>
  <h1>Japan's Shrinking Regions<span class="jp">地域の人口減少</span></h1>
  <p class="lede">When regional decline began, what drives it, whether the drivers have changed, and how far it will go by 2030. Population change is split throughout into natural change (births − deaths) and social change (net migration), because each has its own causes.</p>
  <div class="facts" id="facts"></div>
</header>

<section aria-labelledby="q1">
  <div class="q">Q1 · When did decline start?</div>
  <h2 id="q1">From the periphery inward, in waves set by the economy</h2>
  <p class="answer">The first prefectures peaked around 1980, well before the national peak in 2008. The bubble years pulled young people to Tokyo, and each later wave of decline reached closer to the big cities. Six prefectures were still growing in 2020.</p>

  <div class="chart" id="c-peaks">
    <div class="chart-head"><span class="chart-title">Year each prefecture's population peaked</span><span class="chart-note">Hover a row for details · data from 1975</span></div>
    <div class="svgbox"></div>
  </div>

  <div class="chart" id="c-migration">
    <div class="chart-head"><span class="chart-title">Net migration between prefectures, per 1,000 residents</span><span class="chart-note">社会増減 · moves of Japanese nationals</span></div>
    <div class="legend" data-legend></div>
    <div class="svgbox"></div>
  </div>
  <div class="col stack">
    <p>The Tokyo area's net inflow rose to +5.3 per 1,000 at the bubble peak in 1987 and turned negative in 1994 after the crash. The rest of Japan gained people on net only in those few years. Inflow recovered to +4.4 by 2007, dropped after the Lehman shock, dipped again with COVID in 2021, and was back to +3.2 by 2024.</p>
  </div>

  <figure>
    <img src="__IMG_05_prefecture_growth_heatmap__" alt="Heatmap of 5-year population change by prefecture and period, 1980 to 2020. Rural prefectures turn red from the late 1980s; by 2005 to 2020 38 of 47 prefectures are shrinking, while the Tokyo area, Aichi, Shiga and Okinawa stay blue." loading="lazy">
    <figcaption>Five-year population change by prefecture. Shrinking prefectures per period: 1 (1980–85), 18 (1985–90), 23 (1995–2000), 32 (2000–05), 38 from 2005 on.</figcaption>
  </figure>
</section>

<section aria-labelledby="q2">
  <div class="q">Q2 · What contributes most?</div>
  <h2 id="q2">Age structure sets the death toll; jobs and income steer migration</h2>
  <p class="answer">Natural change is almost fully predictable from today's age structure. Migration, the part policy can influence, follows local income, the job market and the tax base, and is much harder to explain.</p>

  <div class="chart" id="c-coef">
    <div class="chart-head">
      <span class="chart-title">Effect of +1 standard deviation on 5-year change (percentage points, 95% CI)</span>
      <div class="toggle" role="group" aria-label="Which change to show">
        <button type="button" id="coef-social" aria-pressed="true">Migration</button>
        <button type="button" id="coef-natural" aria-pressed="false">Births − deaths</button>
      </div>
    </div>
    <div class="legend" data-legend></div>
    <div class="svgbox"></div>
    <div class="chart-note">Prefectures 1985–2020, standard errors clustered by prefecture. "Between prefectures" compares prefectures within the same period; "within a prefecture" uses only each prefecture's own changes over time.</div>
  </div>

  <div class="col stack">
    <ul class="tight">
      <li><b>Births − deaths:</b> aging rate (−1.92 pp per SD) dominates, then fertility (+0.66). R² is 0.98.</li>
      <li><b>Migration between prefectures:</b> aging (−1.21), income (+0.36), unemployment (−0.36), job openings (+0.25), snow (−0.15).</li>
      <li><b>Migration within a prefecture over time:</b> unemployment (−1.05) is the strongest. When a prefecture's own job market weakens, people leave.</li>
      <li><b>Municipalities:</b> aging (−0.93), fiscal strength (+0.58) and income (+0.53) lead.</li>
      <li>Income, fiscal strength, land price and urban share overlap strongly and act as one "economic core". With 47 prefectures they can't be separated.</li>
    </ul>
  </div>

  <div class="duo">
    <figure>
      <img src="__IMG_14_anova_variance_shares__" alt="ANOVA variance shares. For prefectures, period explains 68 percent of natural change while region explains 32 percent of social change. For municipalities, size class explains more than region." loading="lazy">
      <figcaption>ANOVA: natural change depends mostly on <i>when</i>, migration on <i>where</i>. Among municipalities, size matters more than region. Tukey HSD: Kanto is above, and Hokkaido below, every other region (2015–20).</figcaption>
    </figure>
    <figure>
      <img src="__IMG_08_municipal_change_by_size__" alt="Median municipal population change 2015 to 2020 by size class, split into natural and social change. Municipalities under 5,000 people lose 6.4 percent through natural change and 2.9 percent through migration; cities above 200,000 gain slightly through migration." loading="lazy">
      <figcaption>Small municipalities lose people both ways. Median 2015–20: under 5k −9.4%, over 200k +0.2%.</figcaption>
    </figure>
  </div>
</section>

<section aria-labelledby="q3">
  <div class="q">Q3 · Have the main factors changed?</div>
  <h2 id="q3">Yes: from push by unemployment to pull by income</h2>
  <p class="answer">An F-test rejects "same effects in every period" at both levels (p &lt; 0.0001). Thirty years ago unemployment and harsh winters pushed people out. Today income and the local tax base pull them in, and aging areas keep losing.</p>

  <div class="era">
    <div><b>~30 yrs ago · 1985–90</b><p>Unemployment (−1.60 per SD), aging and snow explain migration between prefectures.</p></div>
    <div><b>~20 yrs ago · 1995–2000</b><p>Income takes over (52% of explained variance). Explanatory power is at its lowest just after the bubble. Municipalities: people leave dense areas for the suburbs (urban share −1.24).</p></div>
    <div><b>~10 yrs ago · 2005–10</b><p>Income (37%) and aging. Fiscal strength becomes the main municipal factor.</p></div>
    <div class="now"><b>Now · 2015–20</b><p>Income (39%) and aging (37%). Unemployment ~1%. Urban share turns positive (+0.23): people are moving back into cities.</p></div>
  </div>

  <div class="chart" id="c-shap">
    <div class="chart-head">
      <span class="chart-title">Share of variance explained, per feature and period (Shapley R²)</span>
      <div class="toggle" role="group" aria-label="Which change to show">
        <button type="button" id="shap-social" aria-pressed="true">Migration</button>
        <button type="button" id="shap-natural" aria-pressed="false">Births − deaths</button>
      </div>
    </div>
    <div class="legend" data-legend></div>
    <div class="svgbox"></div>
    <div class="chart-note">One regression per period across 47 prefectures; bar height = total R². Hover a segment for the coefficient.</div>
  </div>

  <figure>
    <img src="__IMG_20_shapley_by_period_muni__" alt="Shapley R-squared shares by period for municipalities. Aging rate explains most natural change in every period; for migration, fiscal strength becomes the largest share from 2000 on." loading="lazy">
    <figcaption>The same analysis for municipalities, 1995–2020. Migration is far less predictable here (R² 0.12–0.31), because local events such as a new factory or campus aren't in the data.</figcaption>
  </figure>
</section>

<section aria-labelledby="q4">
  <div class="q">Q4 · How far will it go?</div>
  <h2 id="q4">About 5 million fewer people by 2030, mostly outside the big cities</h2>
  <p class="answer" id="fc-answer"></p>

  <div class="chart" id="c-forecast">
    <div class="chart-head"><span class="chart-title">Forecast population change from 2020, by prefecture</span><span class="chart-note">Fukushima excluded (evacuation zones) · hover a row</span></div>
    <div class="legend" data-legend></div>
    <div class="svgbox"></div>
    <details>
      <summary>Show the forecast as a table</summary>
      <div class="tbl"><table id="fc-table"></table></div>
    </details>
  </div>

  <div class="duo">
    <figure>
      <img src="__IMG_16_backtest_pref_2015__" alt="Scatter of predicted versus actual prefecture population change 2015 to 2020, points close to the diagonal; Tokyo grew more than predicted." loading="lazy">
      <figcaption>Backtest: the forecast made with 2015 data against what happened by 2020. Mean error 0.42 pp; naive persistence 0.55 pp.</figcaption>
    </figure>
    <div class="stack col">
      <h3>How much to trust it</h3>
      <ul class="tight">
        <li>Out-of-time backtests on 2005, 2010 and 2015: mean absolute error <b id="mae-best"></b> pp of 5-year change, vs <b id="mae-naive"></b> pp for "the last five years repeat".</li>
        <li>10-year horizon (2010→20): 1.22 pp vs 1.88 pp.</li>
        <li>Live check against the official 2024 estimates: correlation 0.97, but about 0.8 pp too optimistic. Decline sped up after 2020 (birth slump, more deaths, a COVID outflow from Tokyo), and no training period contained that.</li>
        <li>Okinawa began to shrink (−0.1% vs +1.1% predicted), and Tokyo grew less than predicted (+0.9% vs +2.0%).</li>
      </ul>
    </div>
  </div>
</section>

<section aria-labelledby="lim">
  <h2 id="lim">Limitations</h2>
  <ul class="tight limits col">
    <li>The series starts in 1975, so the 1950s–60s rural exodus isn't visible.</li>
    <li>Coefficients are associations, not proven causes. Overlapping economic variables can't be separated with 47 prefectures.</li>
    <li>Distance to metros is measured from prefectural capitals. Municipal migration is the residual of population change minus natural change, because municipal move counts start only in 2018.</li>
    <li>Forecasts use 2020 values and miss the post-2020 acceleration. The 80% intervals rest on one to three backtests. A comparison with the official IPSS projections is still to do.</li>
  </ul>
</section>

<footer>
  <p>Data: Statistics Bureau of Japan, Social and Demographic Statistics System (社会・人口統計体系) via the e-Stat API; prefecture tables 00000101xx–00000102xx, merger-adjusted municipal tables 00000202xx–00000203xx. Population counts are as of 1 October.</p>
  <p>Excluded: Fukushima from 2010 (nuclear evacuation) and Miyake village 1995–2005 (volcanic evacuation). Code, decisions and full tables are in the project repository (<code>reports/REPORT.md</code>).</p>
</footer>
</div>
<div id="tip" hidden></div>

<script>
const D = /*__DATA__*/null;
const NS = "http://www.w3.org/2000/svg";
const tok = v => getComputedStyle(document.documentElement).getPropertyValue(v).trim();
const fmt = (v, d = 1) => (v > 0 ? "+" : v < 0 ? "−" : "") + Math.abs(v).toFixed(d);
const tip = document.getElementById("tip");
function showTip(html, ev) {
  tip.innerHTML = html; tip.hidden = false;
  const w = tip.offsetWidth, h = tip.offsetHeight;
  let x = ev.clientX + 14, y = ev.clientY + 14;
  if (x + w > innerWidth - 8) x = ev.clientX - w - 14;
  if (y + h > innerHeight - 8) y = ev.clientY - h - 14;
  tip.style.left = x + "px"; tip.style.top = y + "px";
}
const hideTip = () => { tip.hidden = true; };
function el(tag, attrs, parent) {
  const e = document.createElementNS(NS, tag);
  for (const k in attrs) e.setAttribute(k, attrs[k]);
  if (parent) parent.appendChild(e);
  return e;
}
function txt(parent, x, y, s, attrs = {}) {
  const t = el("text", { x, y, fill: tok("--ink-2"), "font-size": 11, ...attrs }, parent);
  t.textContent = s; return t;
}
const scale = (d0, d1, r0, r1) => v => r0 + (v - d0) / (d1 - d0) * (r1 - r0);
function ticks(lo, hi, n = 5) {
  const step0 = (hi - lo) / n, mag = Math.pow(10, Math.floor(Math.log10(step0)));
  const step = [1, 2, 2.5, 5, 10].map(m => m * mag).find(s => s >= step0);
  const out = []; for (let v = Math.ceil(lo / step) * step; v <= hi + 1e-9; v += step) out.push(+v.toFixed(6));
  return out;
}
function svgIn(box, w, h) { box.innerHTML = ""; return el("svg", { viewBox: `0 0 ${w} ${h}`, width: w, height: h, role: "img" }, box); }
function legend(chart, items) {
  const L = chart.querySelector("[data-legend]"); if (!L) return;
  L.innerHTML = items.map(([name, color, kind]) =>
    `<span><i class="${kind || ""}" style="background:${color}"></i>${name}</span>`).join("");
}
const EVENTS = [[1991, "Bubble bursts"], [2008, "Lehman shock"], [2011, "Tohoku quake"], [2020, "COVID-19"]];
function events(svg, x, top, bottom, stagger = true) {
  EVENTS.forEach(([yr, lab], i) => {
    el("line", { x1: x(yr), x2: x(yr), y1: top, y2: bottom, stroke: tok("--base"), "stroke-dasharray": "3 3" }, svg);
    txt(svg, x(yr) + 4, top + 10 + (stagger ? (i % 2) * 13 : 0), lab, { fill: tok("--muted"), "font-size": 10 });
  });
}

/* Facts strip */
(function facts() {
  const f = D.facts;
  const items = [
    [String(f.peak_year), `National peak, ${(f.peak_pop / 1e6).toFixed(1)}M people`],
    [String(f.first_nat_dec), "First year deaths exceeded births"],
    [`${f.prefs_declining_now} / 47`, "Prefectures shrinking, 2015–20"],
    [`${Math.round(f.muni_shrinking)}%`, "Municipalities shrinking, 2015–20"],
  ];
  document.getElementById("facts").innerHTML = items.map(([b, s]) => `<div class="fact"><b>${b}</b><span>${s}</span></div>`).join("");
  const drop = (f.pop2020 - f.pop2030) / 1e6;
  document.getElementById("fc-answer").textContent =
    `The 46 prefectures go from ${(f.pop2020 / 1e6).toFixed(1)}M (2020) to a forecast ${(f.pop2030 / 1e6).toFixed(1)}M in 2030, ${drop.toFixed(1)}M fewer people. Akita, Iwate and Aomori lose 12–15%; only Tokyo and Okinawa are forecast to grow. The live check shows actual decline running faster than this.`;
  document.getElementById("mae-best").textContent = f.mae_best.toFixed(2);
  document.getElementById("mae-naive").textContent = f.mae_naive.toFixed(2);
})();

/* Chart: peak year per prefecture */
function drawPeaks() {
  const chart = document.getElementById("c-peaks"), box = chart.querySelector(".svgbox");
  const W = Math.max(box.clientWidth, 340), rows = D.peaks, rh = 17, top = 34, H = top + rows.length * rh + 30;
  const narrow = W < 560, L = 8, R = narrow ? 110 : 150;
  const svg = svgIn(box, W, H), x = scale(1975, 2026, L, W - R);
  ticks(1975, 2025, narrow ? 4 : 6).forEach(v => {
    el("line", { x1: x(v), x2: x(v), y1: top - 6, y2: H - 24, stroke: tok("--grid") }, svg);
    txt(svg, x(v), H - 8, v, { "text-anchor": "middle", fill: tok("--muted") });
  });
  events(svg, x, 0, H - 24);
  rows.forEach((r, i) => {
    const y = top + i * rh + rh / 2;
    el("line", { x1: x(1975), x2: x(r.peak), y1: y, y2: y, stroke: tok("--grid"), "stroke-width": 1.5 }, svg);
    el("circle", { cx: x(r.peak), cy: y, r: 4.5, fill: tok("--s1"), stroke: tok("--surface"), "stroke-width": 2 }, svg);
    txt(svg, x(r.peak) + 8, y + 4, r.name + (r.growing ? " (still growing)" : ""), { "font-size": narrow ? 10 : 11 });
    const hit = el("rect", { x: 0, y: y - rh / 2, width: W, height: rh, fill: "transparent" }, svg);
    hit.addEventListener("pointermove", ev => showTip(`<b>${r.name}</b> · ${r.region}<br>Peak population: ${r.peak}<br>Change since peak: ${fmt(r.since)}%`, ev));
    hit.addEventListener("pointerleave", hideTip);
  });
}

/* Chart: migration lines with crosshair */
function drawMigration() {
  const chart = document.getElementById("c-migration"), box = chart.querySelector(".svgbox");
  const W = Math.max(box.clientWidth, 340), H = 330, narrow = W < 560;
  const m = { l: 40, r: narrow ? 12 : 150, t: 34, b: 28 };
  const colors = [tok("--s1"), tok("--s2"), tok("--s3")];
  legend(chart, D.migration.map((s, i) => [s.name, colors[i]]));
  const all = D.migration.flatMap(s => s.points.map(p => p[1]));
  const lo = Math.floor(Math.min(...all)), hi = Math.ceil(Math.max(...all));
  const svg = svgIn(box, W, H), x = scale(1975, 2024, m.l, W - m.r), y = scale(lo, hi, H - m.b, m.t);
  ticks(lo, hi, 5).forEach(v => {
    el("line", { x1: m.l, x2: W - m.r, y1: y(v), y2: y(v), stroke: v === 0 ? tok("--base") : tok("--grid") }, svg);
    txt(svg, m.l - 6, y(v) + 4, v, { "text-anchor": "end", fill: tok("--muted") });
  });
  ticks(1975, 2024, narrow ? 4 : 6).forEach(v => txt(svg, x(v), H - 8, v, { "text-anchor": "middle", fill: tok("--muted") }));
  events(svg, x, m.t - 22, H - m.b);
  D.migration.forEach((s, i) => {
    el("path", { d: s.points.map((p, j) => (j ? "L" : "M") + x(p[0]).toFixed(1) + "," + y(p[1]).toFixed(1)).join(""),
      fill: "none", stroke: colors[i], "stroke-width": 2, "stroke-linejoin": "round" }, svg);
    if (!narrow) {
      const last = s.points[s.points.length - 1];
      el("line", { x1: x(last[0]) + 6, x2: x(last[0]) + 16, y1: y(last[1]), y2: y(last[1]), stroke: colors[i], "stroke-width": 2 }, svg);
      txt(svg, x(last[0]) + 20, y(last[1]) + 4, s.name);
    }
  });
  const cross = el("line", { y1: m.t, y2: H - m.b, stroke: tok("--muted"), "stroke-width": 1, visibility: "hidden" }, svg);
  const dots = D.migration.map((s, i) => el("circle", { r: 4.5, fill: colors[i], stroke: tok("--surface"), "stroke-width": 2, visibility: "hidden" }, svg));
  const hit = el("rect", { x: m.l, y: m.t, width: W - m.l - m.r, height: H - m.t - m.b, fill: "transparent" }, svg);
  hit.addEventListener("pointermove", ev => {
    const r = svg.getBoundingClientRect(), px = (ev.clientX - r.left) * (W / r.width);
    const yr = Math.round(scale(m.l, W - m.r, 1975, 2024)(px));
    cross.setAttribute("x1", x(yr)); cross.setAttribute("x2", x(yr)); cross.setAttribute("visibility", "visible");
    let html = `<b>${yr}</b>`;
    D.migration.forEach((s, i) => {
      const p = s.points.find(q => q[0] === yr);
      if (p) { dots[i].setAttribute("cx", x(yr)); dots[i].setAttribute("cy", y(p[1])); dots[i].setAttribute("visibility", "visible");
        html += `<br>${s.name}: ${fmt(p[1], 2)}`; }
    });
    showTip(html, ev);
  });
  hit.addEventListener("pointerleave", () => { hideTip(); cross.setAttribute("visibility", "hidden"); dots.forEach(d => d.setAttribute("visibility", "hidden")); });
}

/* Chart: coefficient dot-and-interval */
let coefTarget = "social";
function drawCoef() {
  const chart = document.getElementById("c-coef"), box = chart.querySelector(".svgbox");
  const specs = D.coefs["pref_" + coefTarget];
  const names = { "A: period FE": "Between prefectures", "B: period + prefecture FE": "Within a prefecture over time" };
  const colors = [tok("--s1"), tok("--s2")];
  legend(chart, specs.map((s, i) => [names[s.spec] || s.spec, colors[i], "dot"]));
  const order = specs[0].rows.slice().sort((a, b) => a.coef - b.coef).map(r => r.label);
  const W = Math.max(box.clientWidth, 340), narrow = W < 560, L = narrow ? 150 : 230, R = 16, rh = 34, top = 8;
  const H = top + order.length * rh + 34;
  const all = specs.flatMap(s => s.rows.flatMap(r => [r.lo, r.hi]));
  const lo = Math.min(0, ...all) - 0.1, hi = Math.max(0, ...all) + 0.1;
  const svg = svgIn(box, W, H), x = scale(lo, hi, L, W - R);
  ticks(lo, hi, narrow ? 4 : 6).forEach(v => {
    el("line", { x1: x(v), x2: x(v), y1: top, y2: H - 26, stroke: Math.abs(v) < 1e-9 ? tok("--base") : tok("--grid") }, svg);
    txt(svg, x(v), H - 8, v, { "text-anchor": "middle", fill: tok("--muted") });
  });
  order.forEach((lab, i) => {
    const yc = top + i * rh + rh / 2;
    txt(svg, L - 10, yc + 4, lab, { "text-anchor": "end", "font-size": narrow ? 10 : 11.5 });
    specs.forEach((s, k) => {
      const r = s.rows.find(q => q.label === lab); if (!r) return;
      const y = yc + (k - (specs.length - 1) / 2) * 9;
      el("line", { x1: x(r.lo), x2: x(r.hi), y1: y, y2: y, stroke: colors[k], "stroke-width": 2, "stroke-linecap": "round" }, svg);
      const c = el("circle", { cx: x(r.coef), cy: y, r: 5, fill: colors[k], stroke: tok("--surface"), "stroke-width": 2 }, svg);
      const hit = el("rect", { x: Math.min(x(r.lo), x(r.coef) - 8), y: y - 6, width: Math.max(x(r.hi) - x(r.lo), 16), height: 12, fill: "transparent" }, svg);
      hit.addEventListener("pointermove", ev => showTip(`<b>${lab}</b><br>${names[s.spec] || s.spec}<br>${fmt(r.coef, 2)} pp per SD<br>95% CI ${fmt(r.lo, 2)} to ${fmt(r.hi, 2)}<br>p = ${r.p < 0.001 ? "< 0.001" : r.p.toFixed(3)}`, ev));
      hit.addEventListener("pointerleave", hideTip);
    });
  });
}

/* Chart: Shapley stacked bars */
let shapTarget = "social";
function shapColors() {
  const s = D.shapley.social.series.map(x => x.name), n = D.shapley.natural.series.map(x => x.name);
  const shared = s.filter(v => n.includes(v)), pal = ["--s1", "--s2", "--s3", "--s4", "--s5", "--s6", "--s7", "--s8"].map(tok);
  const map = {};
  [D.shapley.social, D.shapley.natural].forEach(t => {
    const feats = [...shared, ...t.series.map(x => x.name).filter(v => !shared.includes(v))];
    feats.forEach((f, i) => { if (!(f in map) || shared.includes(f)) map[f + "|" + (t === D.shapley.social ? "social" : "natural")] = pal[i]; });
  });
  return { map, shared };
}
function drawShap() {
  const chart = document.getElementById("c-shap"), box = chart.querySelector(".svgbox");
  const t = D.shapley[shapTarget], { map, shared } = shapColors();
  const series = [...t.series.filter(s => shared.includes(s.name)), ...t.series.filter(s => !shared.includes(s.name))];
  const col = s => map[s.name + "|" + shapTarget];
  legend(chart, series.map(s => [s.name, col(s), "band"]));
  const W = Math.max(box.clientWidth, 340), H = 320, m = { l: 40, r: 10, t: 12, b: 44 };
  const svg = svgIn(box, W, H), y = scale(0, 1, H - m.b, m.t);
  const n = t.periods.length, bw = (W - m.l - m.r) / n, barW = Math.min(bw * 0.62, 70);
  ticks(0, 1, 5).forEach(v => {
    el("line", { x1: m.l, x2: W - m.r, y1: y(v), y2: y(v), stroke: v === 0 ? tok("--base") : tok("--grid") }, svg);
    txt(svg, m.l - 6, y(v) + 4, v.toFixed(1), { "text-anchor": "end", fill: tok("--muted") });
  });
  t.periods.forEach((p, i) => {
    const cx = m.l + bw * i + bw / 2; let acc = 0;
    txt(svg, cx, H - m.b + 16, p, { "text-anchor": "middle", fill: tok("--muted"), "font-size": W < 560 ? 9 : 11 });
    txt(svg, cx, H - m.b + 30, "R² " + t.r2[i].toFixed(2), { "text-anchor": "middle", fill: tok("--muted"), "font-size": 9.5 });
    series.forEach(s => {
      const v = s.values[i]; if (v <= 0) return;
      const y0 = y(acc), y1 = y(acc + v), h = Math.max(y0 - y1 - 2, 0.5);
      const r = el("rect", { x: cx - barW / 2, y: y1 + 1, width: barW, height: h, fill: col(s), rx: 2 }, svg);
      r.addEventListener("pointermove", ev => showTip(`<b>${s.name}</b><br>${p}<br>Explains ${(v * 100).toFixed(0)}% of variance (${Math.round(v / t.r2[i] * 100)}% of R²)<br>Effect ${fmt(s.coefs[i], 2)} pp per SD`, ev));
      r.addEventListener("pointerleave", hideTip);
      acc += v;
    });
  });
}

/* Chart: forecast dot + interval */
function drawForecast() {
  const chart = document.getElementById("c-forecast"), box = chart.querySelector(".svgbox");
  const rows = D.forecast, c1 = tok("--s1"), c2 = tok("--s2"), c3 = tok("--s3"), band = tok("--band");
  legend(chart, [["2020→2030 forecast", c1, "dot"], ["80% interval", band, "band"], ["2020→2025 forecast", c2, "dia"], ["Actual 2020→2024", c3, "dot"]]);
  const W = Math.max(box.clientWidth, 340), narrow = W < 560, L = narrow ? 78 : 96, R = 14, rh = 17, top = 6;
  const H = top + rows.length * rh + 34;
  const lo = Math.min(...rows.map(r => r.lo30)) - 0.5, hi = Math.max(...rows.map(r => r.hi30)) + 0.5;
  const svg = svgIn(box, W, H), x = scale(lo, hi, L, W - R);
  ticks(lo, hi, narrow ? 4 : 8).forEach(v => {
    el("line", { x1: x(v), x2: x(v), y1: top, y2: H - 26, stroke: Math.abs(v) < 1e-9 ? tok("--base") : tok("--grid") }, svg);
    txt(svg, x(v), H - 8, (v > 0 ? "+" : "") + v + "%", { "text-anchor": "middle", fill: tok("--muted") });
  });
  rows.slice().reverse().forEach((r, i) => {
    const y = top + i * rh + rh / 2;
    txt(svg, L - 8, y + 4, r.name, { "text-anchor": "end", "font-size": narrow ? 10 : 11 });
    el("line", { x1: x(r.lo30), x2: x(r.hi30), y1: y, y2: y, stroke: band, "stroke-width": 6, "stroke-linecap": "round" }, svg);
    el("circle", { cx: x(r.act24), cy: y, r: 3.5, fill: c3, stroke: tok("--surface"), "stroke-width": 1.5 }, svg);
    const d = 4.2; el("path", { d: `M${x(r.g25)},${y - d} l${d},${d} l${-d},${d} l${-d},${-d}z`, fill: c2, stroke: tok("--surface"), "stroke-width": 1.5 }, svg);
    el("circle", { cx: x(r.g30), cy: y, r: 4.5, fill: c1, stroke: tok("--surface"), "stroke-width": 2 }, svg);
    const hit = el("rect", { x: 0, y: y - rh / 2, width: W, height: rh, fill: "transparent" }, svg);
    hit.addEventListener("pointermove", ev => showTip(`<b>${r.name}</b> · ${(r.pop2020 / 1e3).toFixed(0)}k in 2020<br>2030: ${fmt(r.g30)}% (${fmt(r.lo30)} to ${fmt(r.hi30)}) → ${(r.pop2030 / 1e3).toFixed(0)}k<br>2025: ${fmt(r.g25)}%<br>2020→24 actual ${fmt(r.act24)}% vs predicted ${fmt(r.pred24)}%`, ev));
    hit.addEventListener("pointerleave", hideTip);
  });
}
(function forecastTable() {
  const head = "<tr><th>Prefecture</th><th>2020</th><th>2025 forecast</th><th>2030 forecast</th><th>80% interval 2030</th><th>Actual 2020→24</th></tr>";
  document.getElementById("fc-table").innerHTML = head + D.forecast.map(r =>
    `<tr><td>${r.name}</td><td>${r.pop2020.toLocaleString("en")}</td><td>${fmt(r.g25)}%</td><td>${fmt(r.g30)}%</td><td>${fmt(r.lo30)} to ${fmt(r.hi30)}%</td><td>${fmt(r.act24)}%</td></tr>`).join("");
})();

function bindToggle(a, b, set, draw) {
  const A = document.getElementById(a), B = document.getElementById(b);
  A.addEventListener("click", () => { set("social"); A.setAttribute("aria-pressed", "true"); B.setAttribute("aria-pressed", "false"); draw(); });
  B.addEventListener("click", () => { set("natural"); B.setAttribute("aria-pressed", "true"); A.setAttribute("aria-pressed", "false"); draw(); });
}
bindToggle("coef-social", "coef-natural", v => coefTarget = v, drawCoef);
bindToggle("shap-social", "shap-natural", v => shapTarget = v, drawShap);

const draws = [drawPeaks, drawMigration, drawCoef, drawShap, drawForecast];
const renderAll = () => draws.forEach(f => { try { f(); } catch (e) { console.error(e); } });
renderAll();
let rt; const ro = new ResizeObserver(() => { clearTimeout(rt); rt = setTimeout(renderAll, 120); });
document.querySelectorAll(".chart .svgbox").forEach(b => ro.observe(b));
matchMedia("(prefers-color-scheme: dark)").addEventListener("change", renderAll);
new MutationObserver(renderAll).observe(document.documentElement, { attributes: true, attributeFilter: ["data-theme"] });
</script>

## Step 4: download data from e-Stat

Fetches every variable listed in `config/variables.csv`, at prefecture and municipal level. The municipal
data is the merger-adjusted series (current boundaries). Expect a few minutes.

In [ ]:
run('download.py')

## Step 5: build the 5-year panels

Targets per period, as % of the starting population: total `growth`, `natural` change (births − deaths) and
`social` change (net migration, the residual). Features are measured at the start of each period.
Fukushima is dropped from 2010 (nuclear evacuation), and Miyake village for 1995–2005 (volcanic evacuation).

In [ ]:
run('build_panel.py')

In [ ]:
import pandas as pd
pref = pd.read_parquet('data/processed/pref_panel.parquet')
muni = pd.read_parquet('data/processed/muni_panel.parquet')
print(pref.shape, muni.shape)
pref.groupby('period_label')[['growth', 'natural', 'social']].mean().round(2)

## Step 6: when did decline start? (exploratory analysis)

In [ ]:
run('eda.py', quiet=True)

In [ ]:
from IPython.display import Image, display
for n in ['01_national_population', '02_national_natural_change', '03_prefecture_peak_year', '04_migration_blocs', '05_prefecture_growth_heatmap', '06_natural_vs_social', '07_municipal_shrinking_share', '08_municipal_change_by_size', '09_feature_correlations']:
    display(Image(f'reports/figures/{n}.png'))

## Step 7: feature selection

Within-period Lasso with bootstrap stability selection, random-forest permutation importance and VIF,
separately for natural and social change. This is the slowest step: about 5–10 minutes on Colab.

In [ ]:
run('select_features.py')

In [ ]:
from IPython.display import Image, display
for n in ['10_feature_stability_pref', '11_feature_stability_muni']:
    display(Image(f'reports/figures/{n}.png'))

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open('reports/feature_selection.md').read()))

## Step 8a: what contributes most? (fixed-effects regressions and ANOVA)

In [ ]:
run('models.py', quiet=True)

In [ ]:
from IPython.display import Image, display
for n in ['12_coefficients_pref', '13_coefficients_muni', '14_anova_variance_shares']:
    display(Image(f'reports/figures/{n}.png'))

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open('reports/models_explanatory.md').read()))

## Step 8b: forecasts to 2025 and 2030

Out-of-time backtests (each forecast uses only information available at its start year), compared with
naive persistence; then forecasts from 2020, checked against the official 2024 population estimates.

In [ ]:
run('forecast.py', quiet=True)

In [ ]:
from IPython.display import Image, display
for n in ['15_backtest_errors', '16_backtest_pref_2015', '17_forecast_pref']:
    display(Image(f'reports/figures/{n}.png'))

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open('reports/forecast.md').read()))

## Step 9: do the main factors change over time?

In [ ]:
run('time_varying.py', quiet=True)

In [ ]:
from IPython.display import Image, display
for n in ['18_shapley_by_period_pref', '19_coef_paths_pref_social', '20_shapley_by_period_muni']:
    display(Image(f'reports/figures/{n}.png'))

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open('reports/time_varying.md').read()))

## Step 10: report page and downloads

Builds the interactive report page (`reports/site/index.html`) and zips all results (figures, tables,
report, processed data) so you can download them or copy them to Google Drive.

In [ ]:
run('build_report.py')
!zip -qr results.zip reports data/processed config/selected_features.json
print('Wrote results.zip')
try:
    from google.colab import files
    files.download('results.zip')
except Exception:
    pass

Optional: copy the results to Google Drive instead of downloading them.

```python
from google.colab import drive
drive.mount('/content/drive')
!cp -r reports /content/drive/MyDrive/depopulation-Japan-reports
```